In [ ]:
import os
import importlib.util
sf_options = {"sfUrl":"", "sfUser":"", "sfPassword":"", "sfRole":"", "sfWarehouse":"", "sfDatabase":"", "sfSchema":""}
HANDOFF_MANIFEST = "/local_disk0/tmp/tak861_cached_pilot/handoff_v001/handoff.json"
PILOT_OUTPUT = "/local_disk0/tmp/tak861_cached_pilot/results/v001"
PREVIOUS_PREPARED = ""
ENABLE_NLP = True
MIN_FREE_GIB = 3.0
BOOTSTRAP_DRAWS = 1000
os.environ["OMP_NUM_THREADS"] = "2"
os.environ["OPENBLAS_NUM_THREADS"] = "2"
os.environ["MKL_NUM_THREADS"] = "2"
missing_packages = [name for name in ("numpy", "pandas", "scipy", "sklearn", "lightgbm", "psutil", "matplotlib") if importlib.util.find_spec(name) is None]
if missing_packages:
    raise RuntimeError("Required packages are unavailable: " + ", ".join(missing_packages) + ". Configure the analysis session before running; do not restart a session running extraction.")


In [ ]:
import hashlib, json, types
from pathlib import Path
import numpy as np
import pandas as pd
PILOT_SOURCES = {'inputs': '"""Historical AutoML input construction. Never derives history from snapshot values.\n\nOnly extraction/assembly live here. Every learned transform belongs to TRAIN-only\npreparation. Cached panels are reused by exact patient/date and checked definition\nand content fingerprints. This module does not assert unavailable source lineage.\n"""\nfrom __future__ import annotations\n\nimport hashlib\nimport json\nimport re\nimport time\nimport os\nimport tempfile\nfrom typing import Callable\n\nimport numpy as np\nimport pandas as pd\n\n\nKEYS = ["PATIENT_ID", "AS_OF_DATE"]\nPANEL_MEMORY_LIMIT = 16 * 1024 * 1024\nARRAY_ROW_CHUNK = 2048\n\n\ndef disk_array(shape, dtype=np.float32, prefix="tak861-array-"):\n    """Allocate an owned scratch memmap; callers release it after persistence."""\n    handle, path = tempfile.mkstemp(prefix=prefix, suffix=".npy")\n    os.close(handle)\n    try:\n        value=np.lib.format.open_memmap(path, mode="w+", dtype=dtype, shape=shape)\n    except BaseException:\n        os.unlink(path)\n        raise\n    value._scratch_owner_id=id(value)\n    return value\n\n\ndef _require_array_owner(value):\n    if not isinstance(value,np.memmap) or getattr(value,\'_scratch_owner_id\',None)!=id(value):\n        raise ValueError("Cannot release an array view or unowned mapping; release the owning array")\n\n\ndef readonly_array(value):\n    """Transfer one owned scratch file into a read-only owning mapping."""\n    _require_array_owner(value)\n    path=value.filename\n    value.flush()\n    value._mmap.close()\n    result=np.load(path,mmap_mode=\'r\',allow_pickle=False)\n    result._scratch_owner_id=id(result)\n    value._scratch_owner_id=None\n    return result\n\n\ndef release_array(value):\n    """Release only the explicit scratch owner, never an ndarray/memmap view."""\n    if isinstance(value, np.memmap):\n        _require_array_owner(value)\n        path = value.filename\n        if not value._mmap.closed:\n            value.flush()\n            value._mmap.close()\n        try:\n            os.unlink(path)\n        except FileNotFoundError:\n            pass\n    elif isinstance(value,np.ndarray):\n        current=value.base\n        while isinstance(current,np.ndarray):\n            if isinstance(current,np.memmap):\n                raise ValueError("Cannot release an array view; release the owning array")\n            current=current.base\n\n\nclass DiskPanel:\n    """Ordered metadata plus a disk-backed float64 matrix, never a wide merge.\n\n    Float64 storage preserves the original validated-panel fingerprint; model\n    tensors are still float32, exactly as in the original assembly path.\n    """\n    def __init__(self, metadata, values, names):\n        self.metadata = metadata.reset_index(drop=True)\n        self.values = values\n        self.names = list(names)\n        self.columns = pd.Index(KEYS + ["IS_OBSERVED"] + self.names +\n            [c for c in ("MAX_EVENT_DATE", "MAX_AVAILABLE_DATE", "OBSERVATION_EVIDENCE") if c in metadata])\n        self._validated_specs_hash = None\n        self._panel_hash = None\n        self._panel_cache_key = None\n        self._key_hash = None\n\n    def __len__(self):\n        return len(self.metadata)\n\n    @property\n    def IS_OBSERVED(self):\n        return self.metadata.IS_OBSERVED\n\n    def __contains__(self, key):\n        return key in self.columns\n\n    def __getitem__(self, key):\n        if isinstance(key, str) and key in self.metadata:\n            return self.metadata[key]\n        if isinstance(key, list) and all(k in self.metadata for k in key):\n            return self.metadata[key]\n        raise TypeError("DiskPanel features require bounded iter_frames(), not a full frame copy")\n\n    def iter_frames(self, rows=ARRAY_ROW_CHUNK):\n        for start in range(0, len(self), rows):\n            stop = min(start + rows, len(self))\n            part = self.metadata.iloc[start:stop].reset_index(drop=True).copy()\n            features = pd.DataFrame(self.values[start:stop], columns=self.names, copy=False)\n            yield pd.concat([part, features], axis=1)[list(self.columns)]\n\n    def close(self):\n        release_array(self.values)\nRESERVED_COLUMNS = frozenset({\n    "PATIENT_ID", "AS_OF_DATE", "END_DT", "RESP", "SPLIT", "TARGET", "LABEL",\n    "OUTCOME", "SCORE", "PREDICTION", "IS_OBSERVED", "MAX_EVENT_DATE",\n    "MAX_AVAILABLE_DATE", "OBSERVATION_EVIDENCE", "TIME_STEP", "SNAPSHOT_ROW",\n})\nDATE_POLICIES = ("same_day_months", "month_end_history")\nRECOVERY_QUERY = """SELECT PROCEDURE_CATALOG, PROCEDURE_SCHEMA, PROCEDURE_NAME,\nARGUMENT_SIGNATURE, CREATED, LAST_ALTERED, PROCEDURE_DEFINITION\nFROM DSVC_TAKEDA_TA_PRIVATE.INFORMATION_SCHEMA.PROCEDURES\nWHERE PROCEDURE_SCHEMA IN (\'DS_ML_DEV\', \'DS_ML\', \'DS_ML_PRO\', \'DS_ML_PRD\')\nAND PROCEDURE_NAME IN (\'FEATURE_CREATION_PROD\', \'FEATURE_CREATION_PROC\', \'FEATURE_ENGINEERING_PROD\')\nORDER BY PROCEDURE_NAME, ARGUMENT_SIGNATURE"""\n\n\ndef _require(condition, message):\n    if not condition:\n        raise ValueError(message)\n\n\ndef stable_hash(value):\n    return hashlib.sha256(json.dumps(value, sort_keys=True, separators=(",", ":"),\n                                    allow_nan=False, default=str).encode()).hexdigest()\n\n\ndef _dates(values, label):\n    result = pd.to_datetime(values, errors="raise")\n    _require(result.dt.tz is None, label + ": timezone conversion must be explicit")\n    _require(result.notna().all(), label + ": missing dates")\n    _require(result.eq(result.dt.normalize()).all(), label + ": expected exact dates, not intraday timestamps")\n    return result\n\n\ndef _patients(values):\n    _require(values.notna().all(), "Missing patient keys")\n    \n    out = values.map(str)\n    _require(out.str.len().gt(0).all(), "Empty patient keys")\n    return out\n\n\ndef normalize_snapshots(metadata):\n    """Keep input order exactly; reject duplicate snapshots and cross-split patients."""\n    needed = ["PATIENT_ID", "END_DT", "RESP", "SPLIT"]\n    _require(isinstance(metadata, pd.DataFrame) and not metadata.columns.duplicated().any(),\n             "Snapshot metadata must be a DataFrame with unique columns")\n    _require(set(needed).issubset(metadata.columns), "Metadata requires PATIENT_ID, END_DT, RESP, SPLIT")\n    frame = metadata[needed].copy().reset_index(drop=True)\n    _require(len(frame) > 0, "No snapshots")\n    frame["PATIENT_ID"] = _patients(frame.PATIENT_ID)\n    frame["END_DT"] = _dates(frame.END_DT, "END_DT")\n    _require(not frame.duplicated(["PATIENT_ID", "END_DT"]).any(), "Duplicate snapshot keys")\n    _require(frame.RESP.notna().all() and frame.RESP.isin([0, 1]).all(), "RESP must be binary without missing labels")\n    frame["RESP"] = frame.RESP.astype(np.int64)\n    _require(frame.SPLIT.notna().all(), "Missing split labels")\n    frame["SPLIT"] = frame.SPLIT.astype(str).str.lower().replace({"val": "validation", "valid": "validation"})\n    _require(frame.SPLIT.isin(["train", "validation", "test"]).all(), "Unknown split label")\n    _require(frame.groupby("PATIENT_ID").SPLIT.nunique().eq(1).all(), "Patient appears in multiple splits")\n    return frame\n\n\ndef train_patient_hash(metadata):\n    """The canonical scope fingerprint for any upstream TRAIN-fitted encoding."""\n    meta = normalize_snapshots(metadata)\n    return stable_hash(sorted(meta.loc[meta.SPLIT.eq("train"), "PATIENT_ID"].unique().tolist()))\n\n\ndef verify_fitting_scope(contract, metadata):\n    contract = validate_contract(contract)\n    if contract["fit_scope"] == "train_only_fitted":\n        _require(contract["fitting_patients_sha256"] == train_patient_hash(metadata),\n                 "Upstream learned feature fitting patients differ from current TRAIN patients")\n    return contract\n\n\ndef make_requests(metadata, steps=12, date_policy="same_day_months"):\n    """Return deduplicated extraction keys and a row-aligned snapshot/date grid.\n\n    TIME_STEP zero is newest, preserving the source tensor convention. Older\n    dates are independently derived from the original END_DT, not iterated from\n    previously clamped dates. No label or split is sent to the extractor.\n    """\n    _require(isinstance(steps, int) and 1 <= steps <= 120, "Invalid number of steps")\n    _require(date_policy in DATE_POLICIES, "Unknown historical date policy")\n    meta = normalize_snapshots(metadata)\n    parts = []\n    for step in range(steps):\n        part = meta[["PATIENT_ID", "END_DT"]].copy()\n        part["SNAPSHOT_ROW"] = np.arange(len(meta), dtype=np.int64)\n        part["TIME_STEP"] = step\n        if step == 0:\n            part["AS_OF_DATE"] = part.END_DT\n        elif date_policy == "same_day_months":\n            part["AS_OF_DATE"] = part.END_DT.map(lambda date: date - pd.DateOffset(months=step))\n        else:\n            part["AS_OF_DATE"] = (part.END_DT.dt.to_period("M") - step).dt.to_timestamp(how="end").dt.normalize()\n        parts.append(part)\n    grid = pd.concat(parts, ignore_index=True).sort_values(["SNAPSHOT_ROW", "TIME_STEP"]).reset_index(drop=True)\n    _require(grid.AS_OF_DATE.le(grid.END_DT).all(), "Historical request exceeds snapshot cutoff")\n    requests = grid[KEYS].drop_duplicates().sort_values(KEYS).reset_index(drop=True)\n    return requests, grid\n\n\ndef validate_feature_specs(specs):\n    """Explicit, ordered numeric feature contract; no clinical-name substring filtering."""\n    _require(isinstance(specs, (list, tuple)) and len(specs) > 0, "Feature specifications are required")\n    clean = []\n    for original in specs:\n        _require(isinstance(original, dict), "Feature specification must be a dictionary")\n        item = dict(original)\n        name = item.get("name")\n        _require(isinstance(name, str) and bool(name.strip()), "Missing feature name")\n        _require(name.upper() not in RESERVED_COLUMNS, "Identifier/target/provenance column cannot be a feature: " + name)\n        _require(item.get("kind") in ("count", "binary", "continuous"),\n                 name + ": declare count, binary or continuous; categorical fields require a TRAIN-only encoder")\n        _require(isinstance(item.get("definition"), str) and bool(item["definition"].strip()),\n                 name + ": source feature definition/evidence required")\n        clean.append(item)\n    names = [f["name"] for f in clean]\n    _require(len(names) == len(set(names)), "Duplicate feature names")\n    return clean\n\n\ndef infer_numeric_feature_specs(frame, definitions, exact_exclusions=()):\n    """Create continuous declarations only for explicitly documented numeric fields.\n\n    This does not infer encoding, count or binary semantics from feature names.\n    Definitions must map every selected name to a meaningful source description.\n    Use explicit specs for validated count/binary domains.\n    """\n    excluded = RESERVED_COLUMNS | {str(x).upper() for x in exact_exclusions}\n    selected = [c for c in frame.columns if str(c).upper() not in excluded]\n    specs = []\n    for name in selected:\n        _require(name in definitions, "Missing feature definition: " + name)\n        pd.to_numeric(frame[name], errors="raise")\n        specs.append({"name": name, "kind": "continuous", "definition": definitions[name]})\n    return validate_feature_specs(specs)\n\n\ndef validate_contract(contract):\n    _require(isinstance(contract, dict), "Historical extraction contract is required")\n    result = dict(contract)\n    required = ["source_id", "source_version", "extraction_sha256", "cutoff_rule", "fit_scope", "feature_semantics_evidence"]\n    for key in required:\n        _require(isinstance(result.get(key), str) and bool(result[key].strip()), "Missing extraction contract field: " + key)\n    _require(bool(re.fullmatch(r"[0-9a-f]{64}", result["extraction_sha256"])), "extraction_sha256 must hash actual source extraction code")\n    _require(result["fit_scope"] in ("deterministic_raw", "train_only_fitted"), "Unknown or unsafe AutoML fitting scope")\n    if result["fit_scope"] == "train_only_fitted":\n        _require(bool(re.fullmatch(r"[0-9a-f]{64}", result.get("fitting_patients_sha256", ""))),\n                 "Learned upstream features require fitting_patients_sha256")\n    return result\n\n\ndef _normalize_keys(frame):\n    _require(isinstance(frame, pd.DataFrame) and not frame.columns.duplicated().any(), "Invalid panel/request columns")\n    _require(set(KEYS).issubset(frame.columns), "Historical panel requires PATIENT_ID and AS_OF_DATE")\n    result = frame.copy().reset_index(drop=True)\n    result["PATIENT_ID"] = _patients(result.PATIENT_ID)\n    result["AS_OF_DATE"] = _dates(result.AS_OF_DATE, "AS_OF_DATE")\n    _require(not result.duplicated(KEYS).any(), "Duplicate patient/date panel keys")\n    return result\n\n\ndef validate_panel(panel, specs):\n    specs = validate_feature_specs(specs)\n    if isinstance(panel, DiskPanel):\n        names = [s["name"] for s in specs]\n        _require(names == panel.names, "Disk panel feature order differs")\n        key_frame = _normalize_keys(panel.metadata)\n        _require(key_frame[KEYS].equals(key_frame[KEYS].sort_values(KEYS).reset_index(drop=True)),\n                 "Disk panel keys must be sorted")\n        signature = (stable_hash(specs), frame_hash(panel.metadata))\n        if panel._validated_specs_hash != signature:\n            for part in panel.iter_frames():\n                validate_panel(part, specs)\n            panel._validated_specs_hash = signature\n            panel._panel_hash = None\n            panel._key_hash = frame_hash(panel.metadata[KEYS])\n        return panel\n    frame = _normalize_keys(panel)\n    names = [s["name"] for s in specs]\n    _require(set(names + ["IS_OBSERVED"]).issubset(frame.columns), "Panel lacks feature fields or explicit IS_OBSERVED coverage")\n    _require(frame.IS_OBSERVED.notna().all() and frame.IS_OBSERVED.isin([0, 1, False, True]).all(),\n             "Every panel row requires explicit observation coverage")\n    frame["IS_OBSERVED"] = frame.IS_OBSERVED.astype(bool)\n    observed = frame.IS_OBSERVED.to_numpy()\n    for spec in specs:\n        name = spec["name"]\n        values = pd.to_numeric(frame[name], errors="raise").to_numpy(dtype=np.float64, na_value=np.nan)\n        _require(not np.isinf(values).any(), name + ": infinite feature value")\n        _require(np.isnan(values[~observed]).all(), name + ": unobserved history must remain missing, not copied or zero-filled")\n        finite = values[np.isfinite(values)]\n        if spec["kind"] == "count":\n            _require(np.all((finite >= 0) & (finite == np.floor(finite))), name + ": invalid count values")\n        elif spec["kind"] == "binary":\n            _require(np.isin(finite, [0., 1.]).all(), name + ": invalid binary values")\n        frame[name] = values\n    for col in ("MAX_EVENT_DATE", "MAX_AVAILABLE_DATE"):\n        if col in frame:\n            dates = pd.to_datetime(frame[col], errors="raise")\n            _require(dates.dt.tz is None, col + ": timezone conversion must be explicit")\n            _require(dates.dropna().eq(dates.dropna().dt.normalize()).all(), col + ": expected exact dates")\n            _require((dates.isna() | dates.le(frame.AS_OF_DATE)).all(), col + ": future information at historical cutoff")\n            _require(dates[~observed].isna().all(), col + ": unavailable rows cannot contain event provenance")\n            frame[col] = dates\n    keep = KEYS + ["IS_OBSERVED"] + names + [c for c in ("MAX_EVENT_DATE", "MAX_AVAILABLE_DATE", "OBSERVATION_EVIDENCE") if c in frame]\n    return frame[keep].sort_values(KEYS).reset_index(drop=True)\n\n\ndef canonical_row_bytes(row):\n    canonical = []\n    for value in row:\n        if pd.isna(value):\n            canonical.append(None)\n        elif isinstance(value, (pd.Timestamp, np.datetime64)):\n            canonical.append(pd.Timestamp(value).isoformat())\n        elif isinstance(value, (np.bool_, bool)):\n            canonical.append(bool(value))\n        elif isinstance(value, (float, np.floating)):\n            canonical.append(float(value))\n        elif isinstance(value, (int, np.integer)):\n            canonical.append(int(value))\n        else:\n            canonical.append(str(value))\n    return json.dumps(canonical, separators=(",", ":"), allow_nan=False).encode() + b"\\n"\n\n\ndef frame_hash(frame):\n    """Fingerprint content and ordering without returning any patient-level data."""\n    if isinstance(frame, DiskPanel):\n        \n        \n        \n        cache_key=(tuple(frame.columns),tuple(frame.names),frame_hash(frame.metadata))\n        if frame._panel_hash is not None and frame._panel_cache_key == cache_key:\n            return frame._panel_hash\n    h = hashlib.sha256(json.dumps(list(frame.columns), separators=(",", ":")).encode())\n    if isinstance(frame, DiskPanel):\n        for part in frame.iter_frames():\n            for row in part.itertuples(index=False, name=None):\n                h.update(canonical_row_bytes(row))\n        frame._panel_hash = h.hexdigest()\n        frame._panel_cache_key = cache_key\n        return frame._panel_hash\n    for row in frame.itertuples(index=False, name=None):\n        h.update(canonical_row_bytes(row))\n    return h.hexdigest()\n\n\ndef make_panel_manifest(panel, specs, contract):\n    specs = validate_feature_specs(specs)\n    contract = validate_contract(contract)\n    frame = validate_panel(panel, specs)\n    return {"schema": 1, "contract": contract, "contract_sha256": stable_hash(contract),\n            "feature_specs": specs, "feature_specs_sha256": stable_hash(specs),\n            "panel_sha256": frame_hash(frame), "key_sha256": frame._key_hash if isinstance(frame, DiskPanel) else frame_hash(frame[KEYS]),\n            "rows": len(frame), "observed_rows": int(frame.IS_OBSERVED.sum()),\n            "availability_provenance": "MAX_AVAILABLE_DATE present; nulls remain unresolved" if "MAX_AVAILABLE_DATE" in frame else "Still unresolved: source availability clock not supplied",\n            "event_provenance": "MAX_EVENT_DATE checked where supplied" if "MAX_EVENT_DATE" in frame else "Contract-declared only; event maxima not supplied"}\n\n\ndef obtain_panel(requests, specs, contract, cached_panel=None, cached_manifest=None,\n                 extractor: Callable | None = None):\n    """Reuse checked patient/date rows and extract missing keys once.\n\n    An extractor receives *only* keys and must explicitly return every requested\n    row, including unavailable history. It may not silently omit incomplete rows.\n    The supplied manifest is provenance evidence, not automatic verification that\n    the warehouse extraction really implemented the declared semantics.\n    """\n    started = time.perf_counter()\n    req = _normalize_keys(requests[KEYS]).sort_values(KEYS).reset_index(drop=True)\n    _require(len(req) > 0, "No historical extraction keys")\n    specs = validate_feature_specs(specs)\n    contract = validate_contract(contract)\n    if cached_panel is None:\n        _require(cached_manifest is None, "Manifest supplied without cached values")\n        cached = None\n        missing = req\n    else:\n        _require(isinstance(cached_manifest, dict), "Cached data requires its original definition/content manifest")\n        cached = validate_panel(cached_panel, specs)\n        actual = make_panel_manifest(cached, specs, contract)\n        for key in ("schema", "contract_sha256", "feature_specs_sha256", "panel_sha256", "key_sha256", "rows"):\n            _require(actual[key] == cached_manifest.get(key), "Cached panel mismatch: " + key)\n        missing = req.merge(cached[KEYS], on=KEYS, how="left", indicator=True, validate="one_to_one")\n        missing = missing.loc[missing._merge.eq("left_only"), KEYS].reset_index(drop=True)\n    if len(missing):\n        _require(callable(extractor),\n                 f"Historical AutoML extraction required for {len(missing)} unique patient/date keys. "\n                 "Provide the recovered source query/procedure with exact historical cutoff semantics; a current snapshot table cannot fill history.")\n        fresh = validate_panel(extractor(missing.copy()), specs)\n        coverage = missing.merge(fresh[KEYS], on=KEYS, how="outer", indicator=True, validate="one_to_one")\n        _require(coverage._merge.eq("both").all(), "Extractor must return exactly the requested keys, including explicit unavailable rows")\n        if cached is not None:\n            _require(list(cached.columns) == list(fresh.columns), "Fresh/cached provenance schemas differ; resolve before combining")\n            combined = combine_panels(cached, fresh) if isinstance(cached, DiskPanel) or isinstance(fresh, DiskPanel) else pd.concat([cached, fresh], ignore_index=True)\n        else:\n            combined = fresh\n    else:\n        combined = cached\n    combined = validate_panel(combined, specs)\n    manifest = make_panel_manifest(combined, specs, contract)\n    manifest["extraction_report"] = {\n        "requested_unique_keys": len(req), "cache_keys_reused": len(req) - len(missing),\n        "keys_extracted": len(missing), "panel_cache_rows": len(combined),\n        "elapsed_seconds": time.perf_counter() - started,\n    }\n    return combined, manifest\n\n\ndef combine_panels(left, right):\n    disk = left if isinstance(left, DiskPanel) else right\n    names = disk.names\n    def metadata(part):\n        return part.metadata if isinstance(part, DiskPanel) else part[[c for c in part if c not in names]]\n    meta = pd.concat([metadata(left), metadata(right)], ignore_index=True)\n    order = meta.sort_values(KEYS).index.to_numpy()\n    inverse = np.empty(len(order), dtype=np.int64); inverse[order] = np.arange(len(order))\n    values = disk_array((len(meta),len(names)), np.float64, \'tak861-panel-\')\n    offset = 0\n    for part in (left, right):\n        for start in range(0,len(part),ARRAY_ROW_CHUNK):\n            stop = min(start+ARRAY_ROW_CHUNK,len(part))\n            block = part.values[start:stop] if isinstance(part,DiskPanel) else part.iloc[start:stop][names].to_numpy(dtype=np.float64)\n            values[inverse[offset+start:offset+stop]] = block\n        offset += len(part)\n    values=readonly_array(values)\n    return DiskPanel(meta.iloc[order].reset_index(drop=True),values,names)\n\n\ndef assemble_inputs(metadata, panel, specs, steps=12, date_policy="same_day_months"):\n    """Assemble S=Nx1xF and T=Nx12xF with common ordering and coverage masks."""\n    meta = normalize_snapshots(metadata)\n    specs = validate_feature_specs(specs)\n    frame = validate_panel(panel, specs)\n    requests, grid = make_requests(meta, steps, date_policy)\n    names = [s["name"] for s in specs]\n    panel_keys = frame.metadata[KEYS] if isinstance(frame,DiskPanel) else frame[KEYS]\n    order = pd.MultiIndex.from_frame(panel_keys).get_indexer(pd.MultiIndex.from_frame(grid[KEYS]))\n    _require((order >= 0).all(), "Historical panel does not cover every requested patient/date")\n    x = disk_array((len(meta), steps, len(names)), np.float32, \'tak861-sequence-\')\n    summary = disk_array((len(meta),1,len(names)),np.float32,\'tak861-summary-\')\n    observed = frame.IS_OBSERVED.to_numpy(dtype=bool)\n    mask = observed[order].reshape(len(meta),steps)\n    flat = x.reshape(-1,len(names))\n    try:\n        for start in range(0,len(order),ARRAY_ROW_CHUNK):\n            stop = min(start+ARRAY_ROW_CHUNK,len(order))\n            rows = order[start:stop]\n            block = frame.values[rows].astype(np.float32) if isinstance(frame,DiskPanel) else frame.iloc[rows][names].to_numpy(dtype=np.float32)\n            _require(not np.isinf(block).any(), "Numeric features overflow float32; inspect source scale before modeling")\n            _require(np.isnan(block[~observed[rows]]).all(), "Unobserved steps unexpectedly have values")\n            flat[start:stop] = block\n        for start in range(0,len(meta),ARRAY_ROW_CHUNK):\n            summary[start:start+ARRAY_ROW_CHUNK] = x[start:start+ARRAY_ROW_CHUNK,:1,:]\n        x.flush(); summary.flush()\n    except BaseException:\n        release_array(summary); release_array(x)\n        raise\n    \n    \n    report = {"snapshots": len(meta), "feature_count": len(names), "steps": steps,\n              "unique_patient_dates": len(requests), "sequence_positions": len(grid),\n              "reused_positions": len(grid) - len(requests),\n              "unobserved_steps": int((~mask).sum()),\n              "snapshots_without_current_automl_evidence": int((~mask[:, 0]).sum()),\n              "snapshots_with_incomplete_history": int((~mask).any(axis=1).sum()),\n              "date_policy": date_policy, "orientation": "0=newest",\n              "sequence_kind": "Historical as-of feature summaries; lookbacks may overlap, not disjoint monthly counts",\n              "feature_order_sha256": stable_hash(names),\n              "feature_specs_sha256": stable_hash(specs)}\n    return {"metadata": meta, "y": meta.RESP.to_numpy(dtype=np.int64),\n            "indices": {s: np.flatnonzero(meta.SPLIT.to_numpy() == s) for s in ("train", "validation", "test")},\n            "X_summary": summary, "X_sequence": x,\n            "summary_mask": mask[:, :1].copy(), "sequence_mask": mask,\n            "feature_names": names, "feature_specs": specs,\n            "hashes": {"metadata_sha256": frame_hash(meta), "panel_sha256": frame_hash(frame),\n                       "requests_sha256": frame_hash(requests), "feature_specs_sha256": stable_hash(specs),\n                       "grid_sha256": frame_hash(grid)}, "report": report}\n\n\ndef make_query_extractor(read_query, sql_template, chunk_size=1000):\n    """Bind recovered read-only SQL using a keys-only request CTE.\n\n    sql_template must contain the literal ``{request_sql}`` exactly once, e.g.\n    ``WITH request AS ({request_sql}) SELECT ...``. It must return the standard\n    panel columns. The actual feature calculation is supplied/recovered source\n    code, never an invented fallback. Reader can return pandas or Spark frames.\n    """\n    _require(callable(read_query), "A source query reader is required")\n    _require(isinstance(sql_template, str) and sql_template.count("{request_sql}") == 1,\n             "Recovered SQL must contain exactly one {request_sql} placeholder")\n    _require(isinstance(chunk_size, int) and 1 <= chunk_size <= 10000, "Invalid query chunk size")\n\n    def extractor(requests):\n        req = _normalize_keys(requests[KEYS])\n        outputs = []\n        for start in range(0, len(req), chunk_size):\n            values = []\n            for patient, date in req.iloc[start:start + chunk_size].itertuples(index=False, name=None):\n                \n                patient_sql = patient.replace("\'", "\'\'")\n                values.append(f"SELECT \'{patient_sql}\' AS PATIENT_ID, TO_DATE(\'{date:%Y-%m-%d}\') AS AS_OF_DATE")\n            query = sql_template.replace("{request_sql}", " UNION ALL ".join(values))\n            result = read_query(query)\n            outputs.append(result if isinstance(result, pd.DataFrame) else result.toPandas())\n        _require(outputs, "No historical keys requested")\n        return pd.concat(outputs, ignore_index=True)\n\n    extractor.extraction_sha256 = hashlib.sha256(sql_template.encode()).hexdigest()\n    return extractor\n', 'workflow': '"""Run-scoped orchestration for summary and historical AutoML experiments.\n\nPrivate inputs and row-level predictions stay in the configured artifact store.\nSelection sees TRAIN/VALIDATION only. TEST prediction requires an explicit lock.\nThe notebook injects input/model/evaluation modules and immutable store functions.\n"""\nimport hashlib\nimport io\nimport json\nimport re\nimport time\nimport tempfile\nimport shutil\nimport zipfile\nimport weakref\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\n\ninputs = modeling = comparison = None\nsave_artifacts = read_artifacts = table_exists = None\n\n\ndef plain(value):\n    if isinstance(value, dict):\n        return {str(k): plain(v) for k, v in value.items()}\n    if isinstance(value, (list, tuple)):\n        return [plain(v) for v in value]\n    if isinstance(value, np.ndarray):\n        return plain(value.tolist())\n    if isinstance(value, np.generic):\n        return plain(value.item())\n    if isinstance(value, float) and not np.isfinite(value):\n        return None\n    return value\n\n\ndef json_bytes(value):\n    return json.dumps(plain(value), sort_keys=True, separators=(",", ":"),\n                      allow_nan=False).encode()\n\n\ndef digest(value):\n    return hashlib.sha256(json_bytes(value)).hexdigest()\n\n\ndef array_bytes(value):\n    out = io.BytesIO()\n    np.save(out, np.asarray(value), allow_pickle=False)\n    return out.getvalue()\n\n\ndef array_sha256(value):\n    """Hash the .npy representation without allocating a second full count tensor."""\n    hasher = hashlib.sha256()\n    class Sink:\n        def write(self, block):\n            hasher.update(block)\n            return len(block)\n    np.save(Sink(), np.asarray(value), allow_pickle=False)\n    return hasher.hexdigest()\n\n\ndef bytes_array(value):\n    return np.load(io.BytesIO(value), allow_pickle=False)\n\n\ndef npz_bytes(arrays):\n    out = io.BytesIO()\n    np.savez_compressed(out, **arrays)\n    return out.getvalue()\n\n\ndef prefix(run_id):\n    if not re.fullmatch(r"[A-Z][A-Z0-9_]{0,23}", run_id):\n        raise ValueError("RUN_ID must use uppercase letters, digits and underscores")\n    return "TAK861_AUTOML_SEQUENCE_V1_" + run_id\n\n\ndef assign_patient_split(metadata, seed=42):\n    """Match the documented sorted-patient 70/15/15 stratification algorithm.\n\n    Reproducibility is not an assertion that historical assignments match. If a\n    SPLIT column is supplied, retain and verify it rather than resplitting.\n    """\n    out = metadata.copy()\n    needs_split = "SPLIT" not in out\n    if needs_split:\n        out["SPLIT"] = "train"  \n    out = inputs.normalize_snapshots(out)\n    if needs_split:\n        from sklearn.model_selection import train_test_split\n        patients = out.groupby("PATIENT_ID", sort=True)["RESP"].max()\n        a, rest = train_test_split(patients.index.to_numpy(), test_size=.30,\n                                   random_state=seed, stratify=patients.to_numpy())\n        b, c = train_test_split(rest, test_size=.5, random_state=seed,\n                               stratify=patients.loc[rest].to_numpy())\n        mapping = {p: s for s, values in (("train", a), ("validation", b), ("test", c)) for p in values}\n        out["SPLIT"] = out.PATIENT_ID.map(mapping)\n    if set(out.SPLIT) != {"train", "validation", "test"} or out.SPLIT.isna().any():\n        raise ValueError("Require complete train/validation/test membership")\n    if out.groupby("PATIENT_ID").SPLIT.nunique().max() != 1:\n        raise ValueError("A patient appears in multiple splits")\n    for split, frame in out.groupby("SPLIT"):\n        if set(frame.RESP.unique()) != {0, 1}:\n            raise ValueError("Both classes required in " + split)\n    return out\n\n\nINPUT_NAMES = {"arrays.npz", "metadata.csv", "input_manifest.json"}\n\n\ndef phase_steps(config):\n    phase = config.get("experiment_phase", "TEMPORAL")\n    if phase not in ("SUMMARY", "TEMPORAL"):\n        raise ValueError("Unknown experiment phase")\n    steps = 1 if phase == "SUMMARY" else 12\n    if config.get("raw_automl", {}).get("request_steps", steps) != steps:\n        raise ValueError("Phase and raw extraction length disagree")\n    return steps\n\n\ndef save_inputs(run_prefix, data, panel_manifest, config, implementation_hash):\n    metadata = assign_patient_split(data["metadata"])\n    inputs.verify_fitting_scope(panel_manifest["contract"], metadata)\n    _validate_input_arrays(data, metadata, data["feature_names"], phase_steps(config))\n    metadata["END_DT"] = pd.to_datetime(metadata.END_DT).dt.strftime("%Y-%m-%d")\n    arrays = {key: np.asarray(data[key]) for key in\n              ("X_summary", "X_sequence", "summary_mask", "sequence_mask", "y")}\n    if "X_count" in data:\n        arrays["X_count"] = data["X_count"]\n        arrays["count_mask"] = data["count_mask"]\n    if "X_nlp" in data:\n        arrays["X_nlp"], arrays["nlp_mask"] = data["X_nlp"], data["nlp_mask"]\n    with tempfile.TemporaryDirectory(prefix="tak861_inputs_write_") as directory:\n        arrays_blob = Path(directory) / "arrays.npz"\n        np.savez_compressed(arrays_blob, **arrays)\n        metadata_blob = metadata.to_csv(index=False).encode()\n        manifest = dict(version=1, implementation_hash=implementation_hash,\n                        configuration=plain(config), configuration_hash=digest(config),\n                        panel_manifest=panel_manifest, feature_names=data["feature_names"],\n                        feature_specs=data.get("feature_specs", panel_manifest["feature_specs"]),\n                        input_report=data.get("report", {}),\n                        array_sha256=file_sha256(arrays_blob),\n                        metadata_sha256=hashlib.sha256(metadata_blob).hexdigest(),\n                        count_contract=data.get("count_contract"),\n                        split_counts={s: int((metadata.SPLIT == s).sum()) for s in ("train", "validation", "test")},\n                        prediction_unit="patient_snapshot", private_row_level_artifacts=True)\n        if "X_nlp" in data:\n            manifest["nlp_manifest"] = plain(data["nlp_manifest"])\n        if table_exists(run_prefix + "_INPUTS"):\n            previous_data = load_inputs(run_prefix, implementation_hash)\n            previous = previous_data["manifest"]\n            release_inputs(previous_data)\n            \n            def semantic(value):\n                value = json.loads(json_bytes(value))\n                value["panel_manifest"].pop("extraction_report", None)\n                return value\n            if semantic(previous) != semantic(manifest):\n                raise ValueError("Frozen input source/content/configuration changed; use a new RUN_ID")\n            return previous\n        save_artifacts(run_prefix + "_INPUTS", {"arrays.npz": arrays_blob,\n                       "metadata.csv": metadata_blob, "input_manifest.json": json_bytes(manifest)})\n        return manifest\n\n\ndef file_sha256(path):\n    hasher = hashlib.sha256()\n    with Path(path).open(\'rb\') as stream:\n        for block in iter(lambda: stream.read(1024*1024), b\'\'):\n            hasher.update(block)\n    return hasher.hexdigest()\n\n\nclass _InputDirectory:\n    """Close mappings before scratch cleanup, including notebook restarts/GC."""\n    def __init__(self):\n        self.name = tempfile.mkdtemp(prefix=\'tak861_inputs_read_\')\n        self.mappings = []\n        self.finalizer = weakref.finalize(self, self._remove, self.name, self.mappings)\n\n    @staticmethod\n    def _remove(path, mappings):\n        for mapping in mappings:\n            mapping.close()\n        mappings.clear()\n        shutil.rmtree(path)\n\n    def cleanup(self):\n        self.finalizer()\n\n\ndef release_inputs(data):\n    """Release file-backed raw arrays after a stage; never delete persisted inputs."""\n    temporary = data.pop(\'_input_directory\', None)\n    if temporary is not None:\n        temporary.cleanup()\n    else:\n        for key in (\'X_summary\', \'X_sequence\'):\n            inputs.release_array(data.get(key))\n        modeling.release_array(data.get(\'X_count\'))\n        modeling.release_array(data.get(\'X_nlp\'))\n\n\ndef load_inputs(run_prefix, implementation_hash):\n    temporary = _InputDirectory()\n    data = {}\n    try:\n        artifacts = read_artifacts(run_prefix + "_INPUTS", INPUT_NAMES,\n                                  disk_names={\'arrays.npz\'}, output_dir=temporary.name)\n        manifest = json.loads(artifacts["input_manifest.json"])\n        if manifest["implementation_hash"] != implementation_hash:\n            raise ValueError("Inputs belong to a different notebook implementation; use a new RUN_ID")\n        if (file_sha256(artifacts[\'arrays.npz\']) != manifest[\'array_sha256\'] or\n                hashlib.sha256(artifacts[\'metadata.csv\']).hexdigest() != manifest[\'metadata_sha256\']):\n            raise ValueError(\'Input content changed\')\n        allowed = {\'X_summary\',\'X_sequence\',\'X_count\',\'summary_mask\',\'sequence_mask\',\'count_mask\',\'y\',\'X_nlp\',\'nlp_mask\'}\n        with zipfile.ZipFile(artifacts[\'arrays.npz\']) as archive:\n            for name in archive.namelist():\n                key = name[:-4] if name.endswith(\'.npy\') else \'\'\n                if key not in allowed or key in data:\n                    raise ValueError(\'Unexpected or duplicate input array\')\n                target = Path(temporary.name) / name\n                with archive.open(name) as source, target.open(\'wb\') as dest:\n                    shutil.copyfileobj(source, dest, length=1024*1024)\n                data[key] = np.load(target, mmap_mode=\'r\', allow_pickle=False)\n                temporary.mappings.append(data[key]._mmap)\n        metadata = pd.read_csv(io.BytesIO(artifacts["metadata.csv"]), dtype={"PATIENT_ID": str}, keep_default_na=False)\n        metadata = assign_patient_split(metadata)\n        inputs.verify_fitting_scope(manifest["panel_manifest"]["contract"], metadata)\n        data.update(metadata=metadata, manifest=manifest, feature_names=manifest["feature_names"],\n                    feature_specs=manifest["feature_specs"], _input_directory=temporary,\n                    indices={s: np.flatnonzero(metadata.SPLIT.to_numpy() == s) for s in ("train", "validation", "test")})\n        if "nlp_manifest" in manifest:\n            data["nlp_manifest"] = manifest["nlp_manifest"]\n        _validate_input_arrays(data, metadata, manifest["feature_names"], phase_steps(manifest["configuration"]))\n        return data\n    except BaseException:\n        data[\'_input_directory\'] = temporary\n        release_inputs(data)\n        raise\n\n\ndef _validate_input_arrays(data, metadata, feature_names, sequence_steps=12):\n    n, f = len(metadata), len(feature_names)\n    if f == 0 or len(set(feature_names)) != f:\n        raise ValueError("Invalid AutoML feature order")\n    for value, mask, steps in (("X_summary", "summary_mask", 1), ("X_sequence", "sequence_mask", sequence_steps)):\n        x, valid = np.asarray(data[value]), np.asarray(data[mask])\n        if x.shape != (n, steps, f) or valid.shape != (n, steps) or valid.dtype != bool:\n            raise ValueError("Invalid AutoML array/mask shape: " + value)\n        for start in range(0,n,256):\n            block, mask = x[start:start+256], valid[start:start+256]\n            if np.isinf(block).any() or not np.isnan(block[~mask]).all():\n                raise ValueError("Invalid feature values or missing-history padding")\n    for start in range(0,n,256):\n        if not np.array_equal(data["X_summary"][start:start+256], data["X_sequence"][start:start+256, :1], equal_nan=True):\n            raise ValueError("Summary inputs differ from historical current snapshot")\n    if not np.array_equal(data["summary_mask"], data["sequence_mask"][:, :1]):\n        raise ValueError("Invalid current snapshot observation mask")\n    if np.asarray(data["y"]).shape != (n,) or not np.array_equal(data["y"], metadata.RESP.to_numpy()):\n        raise ValueError("Saved labels do not align with metadata")\n    if "X_count" in data:\n        if np.asarray(data["X_count"]).shape != (n, 12, 1028) or np.asarray(data["count_mask"]).shape != (n, 12):\n            raise ValueError("Count reference shape mismatch")\n        if np.asarray(data["count_mask"]).dtype != bool:\n            raise ValueError("Invalid count mask or nonfinite count values")\n        for start in range(0,n,256):\n            if not np.isfinite(data["X_count"][start:start+256]).all():\n                raise ValueError("Invalid count mask or nonfinite count values")\n    if "X_nlp" in data:\n        names = data.get("nlp_manifest", {}).get("feature_names", [])\n        if not names or len(set(names)) != len(names):\n            raise ValueError("Missing or invalid frozen NLP feature order")\n        if np.shape(data["X_nlp"]) != (n, 12, len(names)) or np.shape(data["nlp_mask"]) != (n, 12):\n            raise ValueError("NLP array/mask shape mismatch")\n        if np.asarray(data["nlp_mask"]).dtype != bool:\n            raise ValueError("NLP activity mask must be boolean")\n        for start in range(0, n, 256):\n            block = data["X_nlp"][start:start+256]\n            mask = data["nlp_mask"][start:start+256]\n            if not np.isfinite(block).all() or (block < 0).any() or (block[~mask] != 0).any():\n                raise ValueError("NLP values must be finite nonnegative with zero inactive steps")\n\n\ndef attach_count_reference(data, values, metadata, contract, valid=None):\n    """Optional verified reuse, aligned by exact keys and independently hashed.\n\n    A provenance declaration limits claims; it is not evidence of an unknown\n    historical extraction rule. Never rename a different universe to 1,028.\n    """\n    if not isinstance(contract, dict) or not all(contract.get(k) for k in\n            ("source_id", "source_version", "content_sha256", "feature_names", "provenance_status",\n             "orientation", "date_policy", "mask_kind", "mask_sha256", "asof_grid_sha256")):\n        raise ValueError("Count reuse needs source/version/content, calendar, explicit mask and provenance fingerprints")\n    values = np.asanyarray(values)\n    names = contract["feature_names"]\n    if values.ndim != 3 or values.shape != (len(metadata), 12, 1028) or len(names) != 1028 or len(set(names)) != 1028:\n        raise ValueError("Count reference must preserve the complete 1,028 candidate universe")\n    if contract.get("n_steps") != 12 or contract["orientation"] != "0=newest" or contract["date_policy"] not in inputs.DATE_POLICIES:\n        raise ValueError("Count cache calendar orientation/length must be explicit")\n    valid = np.asarray(valid)\n    if valid.shape != values.shape[:2] or valid.dtype != bool:\n        raise ValueError("Count reuse requires an explicit boolean position mask")\n    if array_sha256(valid) != contract["mask_sha256"]:\n        raise ValueError("Count position mask hash mismatch")\n    if array_sha256(values) != contract["content_sha256"]:\n        raise ValueError("Count cache content hash mismatch")\n    for block in np.array_split(values, max(1, len(values) // 512)):\n        if not np.isfinite(block).all() or (block < 0).any():\n            raise ValueError("Invalid count values")\n        finite = block[np.isfinite(block)]\n        if (finite != np.floor(finite)).any():\n            raise ValueError("Fractional count values")\n        if np.any(finite.astype(np.float32).astype(np.float64) != finite):\n            raise ValueError("Count loses exact float32 precision")\n    old = inputs.normalize_snapshots(metadata)\n    target = inputs.normalize_snapshots(data["metadata"])\n    if contract.get("metadata_sha256") != inputs.frame_hash(old):\n        raise ValueError("Count cache snapshot metadata/order fingerprint mismatch")\n    _, old_grid = inputs.make_requests(old, date_policy=contract["date_policy"])\n    if inputs.frame_hash(old_grid) != contract["asof_grid_sha256"]:\n        raise ValueError("Count cache date grid mismatch")\n    target_policy = data.get("report", {}).get("date_policy", data.get("manifest", {}).get("input_report", {}).get("date_policy"))\n    if target_policy != contract["date_policy"]:\n        raise ValueError("Count and AutoML historical date policies differ; do not concatenate misaligned dates")\n    old["_row"] = np.arange(len(old))\n    joined = target.merge(old[["PATIENT_ID", "END_DT", "RESP", "_row"]],\n                          on=["PATIENT_ID", "END_DT"], how="left", validate="one_to_one", suffixes=("", "_cached"))\n    if joined._row.isna().any() or not joined.RESP.eq(joined.RESP_cached).all() or len(old) != len(target):\n        raise ValueError("Count cache cohort/label mismatch")\n    order = joined._row.to_numpy(dtype=int)\n    if values.dtype == np.float32 and np.array_equal(order,np.arange(len(values))):\n        data["X_count"] = values  \n    else:\n        data["X_count"] = modeling.take_rows(values,order,dtype=np.float32)\n    data["count_mask"] = valid[order].copy()\n    data["count_contract"] = dict(contract, coverage_note="Position mask semantics are explicit; a saved row does not establish clinical observation coverage")\n    return data\n\n\nPREP_NAMES = {"preparation.json", "protocol.json"}\n\n\ndef prepare_run(data, run_prefix, config, implementation_hash):\n    steps = phase_steps(config)\n    if data["X_sequence"].shape[1] != steps:\n        raise ValueError("AutoML history length differs from the registered experiment phase")\n    if data["manifest"]["panel_manifest"]["contract"]["fit_scope"] != "deterministic_raw":\n        raise ValueError("This fold-local selection protocol requires raw deterministic AutoML inputs. "\n                         "A transform or vocabulary selected using all TRAIN labels is not fold-local. "\n                         "Recover raw extraction without RESP-based fitting, or implement upstream reconstruction inside each fold.")\n    table = run_prefix + "_PREPARATION"\n    expected = dict(input_manifest_hash=digest(data["manifest"]), implementation_hash=implementation_hash,\n                    config_hash=digest(config))\n    if table_exists(table):\n        saved = read_artifacts(table, PREP_NAMES)\n        prep, protocol = json.loads(saved["preparation.json"]), json.loads(saved["protocol.json"])\n        if protocol["identity"] != expected:\n            raise ValueError("Preparation identity changed; use a new RUN_ID")\n        return prep, protocol\n    ti = data["indices"]["train"]\n    groups = data["metadata"].PATIENT_ID.to_numpy()[ti]\n    primary_fraction = float(config.get("primary_lift_fraction", .10))\n    if not np.isfinite(primary_fraction) or not 0 < primary_fraction <= 1:\n        raise ValueError("primary_lift_fraction must be in (0,1]")\n    selection_config, model_config = dict(config.get("selection", {})), dict(config.get("model", {}))\n    for settings in (selection_config, model_config):\n        if "primary_lift_fraction" in settings and settings["primary_lift_fraction"] != primary_fraction:\n            raise ValueError("Conflicting primary capacity; set primary_lift_fraction once in the top-level configuration")\n        settings["primary_lift_fraction"] = primary_fraction\n    declared_kinds = [s["kind"] for s in data["manifest"]["feature_specs"]]\n    if config.get("feature_kinds", declared_kinds) != declared_kinds:\n        raise ValueError("Model preprocessing kinds differ from the recovered source feature contract")\n    prep = {"automl": modeling.prepare_features(\n        data["X_summary"], data["X_sequence"], data["y"][ti], groups,\n        data["feature_names"], valid_train=data["sequence_mask"], row_indices=ti,\n        feature_kinds=declared_kinds, config=selection_config, sequence_steps=steps)}\n    arms = {"SUMMARY": dict(family="transformer", input="X_summary", mask="summary_mask", preparation="automl"),\n            "SEQUENCE": dict(family="transformer", input="X_sequence", mask="sequence_mask", preparation="automl")}\n    if "X_count" in data:\n        count = data["X_count"]\n        prep["count"] = modeling.prepare_features(count[:, :1], count, data["y"][ti], groups,\n            data["manifest"]["count_contract"]["feature_names"], valid_train=data["count_mask"], row_indices=ti,\n            feature_kinds=["count"] * count.shape[-1], config=selection_config)\n        arms["COUNT"] = dict(family="transformer", input="X_count", mask="count_mask", preparation="count")\n        arms["STATIC_FUSION"] = dict(arms["COUNT"], static_input="X_summary", static_mask="summary_mask",\n                                    static_preparation="automl", experiment="count_sequence_plus_snapshot_automl")\n        arms["TEMPORAL_FUSION"] = dict(arms["COUNT"], temporal_input="X_sequence", temporal_mask="sequence_mask",\n                                      temporal_preparation="automl", experiment="count_plus_historical_automl_per_timestep")\n    if config.get("require_count_reference", True) and "COUNT" not in arms:\n        raise ValueError("Both requested incremental-value experiments require the matched 1,028-count reference")\n    if not config.get("include_automl_only_controls", False):\n        arms = {name: arm for name, arm in arms.items() if name not in ("SUMMARY", "SEQUENCE")}\n    if steps == 1:\n        arms = {name: arm for name, arm in arms.items() if name not in ("SEQUENCE", "TEMPORAL_FUSION")}\n    if config.get("include_nlp", False):\n        contract = data.get("nlp_manifest", data["manifest"].get("nlp_manifest", {}))\n        if ("X_nlp" not in data or "COUNT" not in arms or contract.get("fit_scope") != "train_only"\n                or contract.get("orientation") != "0=newest"\n                or contract.get("metadata_hash") != inputs.frame_hash(data["metadata"])\n                or contract.get("train_patients_sha256") != inputs.train_patient_hash(data["metadata"])):\n            raise ValueError("NLP augmentation needs aligned TRAIN-only monthly inputs and count reference")\n        prep["nlp"] = dict(feature_names=contract["feature_names"], manifest_hash=digest(contract),\n                           selection="fixed TRAIN-only TF-IDF vocabulary; no supervised screening",\n                           scaling="Per-document L2-normalized sublinear TF-IDF; no second fitted scaler")\n        arms["NLP_FUSION"] = dict(arms["COUNT"], nlp_input="X_nlp", nlp_mask="nlp_mask",\n                                  experiment="count_plus_monthly_train_fitted_tfidf")\n    if config.get("include_lightgbm", False):\n        raise ValueError("LightGBM refit is deferred in this bounded two-experiment protocol")\n    seeds = config.get("seeds", [42, 142, 242])\n    if not seeds or len(set(seeds)) != len(seeds) or any(type(s) is not int or s < 0 for s in seeds):\n        raise ValueError("Seeds must be a prespecified unique nonempty list")\n    protocol = dict(identity=expected, arms=arms, seeds=seeds, preparation_hash=digest(prep),\n                    model_config=model_config, primary_metric="lift_at_primary_capacity",\n                    primary_lift_fraction=primary_fraction,\n                    primary_capacity_provenance="configured before fit; 10% default is an operating assumption, not a recovered prior agreement",\n                    lift_population="patient_snapshots; not unique-patient interventions",\n                    lift_tie_policy="score descending then frozen snapshot row order; exact ceil(fraction*N)",\n                    selection_split="train_grouped_cv_and_validation", test_used_for_selection=False,\n                    top_fractions=sorted({.1, .2, .3, primary_fraction}),\n                    test_evaluation_ordinal=None if \'client_universe\' in config else 3,\n                    legacy_test_evaluations=2,\n                    test_scope=("Original frozen cohort; previously inspected TEST patients, not an untouched holdout"\n                                if config.get(\'client_universe\',{}).get(\'table_name\')==\'TAK861_TX_READY_V63_MODEL_DATA\' else\n                                "Revised universe includes previously inspected TEST patients and possibly new patients; not an untouched holdout"\n                                if \'client_universe\' in config else "retrospective; this cohort already evaluated twice"),\n                    partial_seed_rule="all completed runs shown; incomplete protocol cannot be locked",\n                    comparison_scope="matched count control versus static and historical AutoML augmentation; descriptive development evidence")\n    if "experiment_phase" in config:\n        protocol["experiment_phase"] = config["experiment_phase"]\n        protocol["automl_proxy_scope"] = ("single-anchor: historical mean equals current value; SD and slope zero"\n                                           if steps == 1 else "12-anchor summary, mean, SD and slope descriptors")\n        protocol["nlp_contrast_scope"] = "NLP plus monthly history versus AutoML representation; not isolated language or architecture effect"\n        protocol["comparison_scope"] = ("Within-phase development comparison; SUMMARY and TEMPORAL have different feature-selection scope. "\n                                         "Do not rank cross-phase predictions or claim independent production superiority.")\n    save_artifacts(table, {"preparation.json": json_bytes(prep), "protocol.json": json_bytes(protocol)})\n    return prep, protocol\n\n\nRUN_NAMES = {"model.bin", "run.json", "train.npy", "validation.npy"}\n\n\ndef pack_run(run, identity):\n    summary = {k: v for k, v in run.items() if k not in ("model_blob", "train_scores", "val_scores", "validation_scores", "model")}\n    blobs = {"model.bin": run["model_blob"], "train.npy": array_bytes(run["train_scores"]),\n             "validation.npy": array_bytes(run["val_scores"])}\n    summary.update(identity=identity, payload_hashes={k: hashlib.sha256(v).hexdigest() for k, v in blobs.items()})\n    blobs["run.json"] = json_bytes(summary)\n    return blobs\n\n\ndef unpack_run(artifacts, identity):\n    summary = json.loads(artifacts["run.json"])\n    if summary["identity"] != identity:\n        raise ValueError("Cached fitted model identity changed")\n    for name, expected in summary["payload_hashes"].items():\n        if hashlib.sha256(artifacts[name]).hexdigest() != expected:\n            raise ValueError("Fitted model artifact hash mismatch")\n    return dict(summary, model_blob=artifacts["model.bin"], train_scores=bytes_array(artifacts["train.npy"]),\n                val_scores=bytes_array(artifacts["validation.npy"]))\n\n\ndef arm_arrays(data, prep, arm, split, subset=None):\n    idx = data[\'indices\'][split]\n    if subset is not None:\n        idx = idx[subset]\n    valid = data[arm[\'mask\']][idx]\n    x = static = transformed = temporal = merged = None\n    try:\n        x = modeling.transform_inputs(data[arm[\'input\']], prep[arm[\'preparation\']],\n                                      valid=data[arm[\'mask\']], row_indices=idx)\n        if \'static_input\' in arm:\n            mask = data[arm[\'static_mask\']][idx]\n            transformed = modeling.transform_inputs(data[arm[\'static_input\']], prep[arm[\'static_preparation\']],\n                                                    valid=data[arm[\'static_mask\']], row_indices=idx)\n            static = modeling.allocate_array((len(idx), transformed.shape[-1]+1), np.float32)\n            for start in range(0,len(idx),256):\n                static[start:start+256,:-1] = transformed[start:start+256,0]\n                static[start:start+256,-1:] = mask[start:start+256]\n            modeling.release_array(transformed)\n            transformed = None\n        if \'temporal_input\' in arm:\n            mask = data[arm[\'temporal_mask\']][idx]\n            temporal = modeling.transform_inputs(data[arm[\'temporal_input\']], prep[arm[\'temporal_preparation\']],\n                                                 valid=data[arm[\'temporal_mask\']], row_indices=idx)\n            if temporal.shape[:2] != x.shape[:2]:\n                raise ValueError(\'Count and AutoML monthly shapes disagree\')\n            merged = modeling.allocate_array((*x.shape[:2], x.shape[-1]+temporal.shape[-1]+1),np.float32)\n            k=x.shape[-1]\n            for start in range(0,len(idx),256):\n                merged[start:start+256,:,:k] = x[start:start+256]\n                merged[start:start+256,:,k:-1] = temporal[start:start+256]\n                merged[start:start+256,:,-1] = mask[start:start+256]\n            modeling.release_array(x);modeling.release_array(temporal)\n            x,merged,temporal = merged,None,None\n        if \'nlp_input\' in arm:\n            raw_nlp = data[arm[\'nlp_input\']]\n            activity = data[arm[\'nlp_mask\']]\n            if raw_nlp.shape[1] != x.shape[1]:\n                raise ValueError(\'NLP monthly grid length mismatch\')\n            merged = modeling.allocate_array((*x.shape[:2], x.shape[-1]+raw_nlp.shape[-1]+1), np.float32)\n            k = x.shape[-1]\n            for start in range(0, len(idx), 256):\n                rows = idx[start:start+256]\n                merged[start:start+256, :, :k] = x[start:start+256]\n                merged[start:start+256, :, k:-1] = raw_nlp[rows]\n                merged[start:start+256, :, -1] = activity[rows]\n            modeling.release_array(x)\n            x, merged = merged, None\n        return x,valid,static\n    except BaseException:\n        for value in (x,static,transformed,temporal,merged):\n            modeling.release_array(value)\n        raise\n\n\ndef train_registered(data, prep, protocol, run_prefix):\n    if digest(prep) != protocol["preparation_hash"] or digest(data["manifest"]) != protocol["identity"]["input_manifest_hash"]:\n        raise ValueError("Training inputs or preparation changed")\n    groups = data["metadata"].PATIENT_ID.to_numpy()\n    ti, vi = data["indices"]["train"], data["indices"]["validation"]\n    results = {}\n    for name, arm in protocol["arms"].items():\n        train = validation = static_train = static_val = None\n        try:\n            train, train_mask, static_train = arm_arrays(data, prep, arm, "train")\n            validation, val_mask, static_val = arm_arrays(data, prep, arm, "validation")\n            results[name] = []\n            for seed in protocol["seeds"]:\n                identity = dict(protocol_hash=digest(protocol), arm=name, seed=seed)\n                table = run_prefix + "_" + name + "_S" + str(seed)\n                if not table_exists(table):\n                    if arm["family"] != "transformer":\n                        raise ValueError("Only Transformer arms are registered; LightGBM remains deferred")\n                    trainer = modeling.train_encoder\n                    run = trainer(train, data["y"][ti], groups[ti], validation, data["y"][vi],\n                                  valid_train=train_mask, valid_val=val_mask,\n                                  static_train=static_train, static_val=static_val,\n                                  groups_val=groups[vi],\n                                  config=protocol["model_config"], seed=seed)\n                    save_artifacts(table, pack_run(run, identity))\n                result = unpack_run(read_artifacts(table, RUN_NAMES), identity)\n                if len(result["train_scores"]) != len(ti) or len(result["val_scores"]) != len(vi):\n                    raise ValueError("Prediction length does not match frozen split")\n                results[name].append(result)\n                print(name, "seed", seed, "completed; validation lift",\n                      modeling.lift_at_fraction(data["y"][vi], result["val_scores"], protocol["primary_lift_fraction"]),\n                      "secondary AP", comparison.binary_metrics(data["y"][vi], result["val_scores"])["average_precision"])\n        finally:\n            for array in (train,validation,static_train,static_val):\n                modeling.release_array(array)\n    return results\n\n\ndef read_runs(protocol, run_prefix):\n    return {name: [unpack_run(read_artifacts(run_prefix + "_" + name + "_S" + str(seed), RUN_NAMES),\n                  dict(protocol_hash=digest(protocol), arm=name, seed=seed)) for seed in protocol["seeds"]]\n            for name in protocol["arms"]}\n\n\ndef make_lock(data, prep, protocol, results, run_prefix):\n    if set(results) != set(protocol["arms"]) or digest(prep) != protocol["preparation_hash"]:\n        raise ValueError("Incomplete registered experiments")\n    vi, ti = data["indices"]["validation"], data["indices"]["train"]\n    models, metrics = {}, {}\n    for name, runs in results.items():\n        if len(runs) != len(protocol["seeds"]) or [r["identity"]["seed"] for r in runs] != protocol["seeds"]:\n            raise ValueError("All prespecified seeds must complete before locking")\n        val = np.mean([r["val_scores"] for r in runs], axis=0)\n        train = np.mean([r["train_scores"] for r in runs], axis=0)\n        policy = comparison.select_validation_threshold(data["y"][vi], val)\n        models[name] = dict(predictor_hashes=[r["payload_hashes"] for r in runs], threshold=policy)\n        metrics[name] = dict(train=comparison.binary_metrics(data["y"][ti], train, policy,\n                                                             top_fractions=protocol["top_fractions"]),\n                             validation=comparison.binary_metrics(data["y"][vi], val, policy,\n                                                                  top_fractions=protocol["top_fractions"]))\n        for split_name, yy, ss in (("train", data["y"][ti], train), ("validation", data["y"][vi], val)):\n            metrics[name][split_name]["primary_lift"] = modeling.lift_at_fraction(yy, ss, protocol["primary_lift_fraction"])\n            metrics[name][split_name]["primary_lift_fraction"] = protocol["primary_lift_fraction"]\n    lock = dict(protocol_hash=digest(protocol), preparation_hash=digest(prep),\n                input_manifest_hash=digest(data["manifest"]), models=models, metrics=metrics,\n                test_prediction_performed=False, test_use_status_note="False at immutable lock creation; completed TEST use is recorded in FINAL_TEST report",\n                test_scope=protocol["test_scope"],\n                primary_metric=protocol["primary_metric"], primary_lift_fraction=protocol["primary_lift_fraction"],\n                threshold_note="Validation F1 threshold is secondary; primary lift uses the registered capacity, not this threshold")\n    save_artifacts(run_prefix + "_LOCK", {"lock.json": json_bytes(lock)})\n    return lock\n\n\ndef paired_lift_bootstrap(y, scores_a, scores_b, patient_ids, fractions=(.1, .2, .3),\n                          n_bootstrap=1000, seed=20261013, confidence=.95):\n    """Paired patient resampling for snapshot lift, with exact repeated-row capacity.\n\n    Patient multiplicities repeat every snapshot of a sampled patient. Ranking\n    ties use frozen original row order in every draw. Integer multiplicities\n    avoid materializing duplicate rows; a boundary row may be selected partly.\n    """\n    y, a, b, groups = [np.asarray(v) for v in (y, scores_a, scores_b, patient_ids)]\n    fractions = sorted(set(float(f) for f in fractions))\n    if (y.ndim != 1 or any(v.shape != y.shape for v in (a, b, groups))\n            or set(np.unique(y)) != {0, 1} or not np.isfinite(a).all() or not np.isfinite(b).all()\n            or not fractions or any(not np.isfinite(f) or not 0 < f <= 1 for f in fractions)\n            or type(n_bootstrap) is not int or n_bootstrap < 20 or not 0 < confidence < 1):\n        raise ValueError("Paired lift bootstrap requires aligned two-class data, valid fractions and at least 20 draws")\n    if any(v is None or str(v) in ("", "nan") for v in groups):\n        raise ValueError("Patient IDs must be complete for clustered lift uncertainty")\n    unique, inverse = np.unique(groups.astype(str), return_inverse=True)\n    orders = [np.lexsort((np.arange(len(y)), -scores.astype(float))) for scores in (a, b)]\n    def lifts(order, multiplicities):\n        counts = multiplicities[order]\n        before = np.cumsum(counts) - counts\n        n = int(multiplicities.sum())\n        prevalence = float(np.dot(y, multiplicities) / n)\n        values = []\n        for fraction in fractions:\n            k = max(1, int(np.ceil(fraction*n)))\n            taken = np.minimum(counts, np.maximum(k-before, 0))\n            values.append(float(np.dot(y[order], taken) / k / prevalence))\n        return np.asarray(values)\n    points = lifts(orders[0], np.ones(len(y), dtype=int)) - lifts(orders[1], np.ones(len(y), dtype=int))\n    rng, draws, invalid = np.random.default_rng(seed), [], 0\n    for _ in range(n_bootstrap):\n        sampled = rng.integers(0, len(unique), size=len(unique))\n        multiplicities = np.bincount(sampled, minlength=len(unique))[inverse]\n        positives, total = int(np.dot(y, multiplicities)), int(multiplicities.sum())\n        if positives == 0 or positives == total:\n            invalid += 1\n            continue\n        draws.append(lifts(orders[0], multiplicities)-lifts(orders[1], multiplicities))\n    reliable = len(draws) >= max(20, int(np.ceil(.8*n_bootstrap)))\n    values = np.asarray(draws)\n    tail = (1-confidence)/2\n    by_fraction = {}\n    for j, fraction in enumerate(fractions):\n        lower, upper = (np.quantile(values[:, j], [tail, 1-tail]).tolist() if reliable else (None, None))\n        by_fraction[str(fraction)] = dict(fraction=fraction, delta=float(points[j]), lower=lower, upper=upper,\n                                           excludes_zero=bool(reliable and (lower > 0 or upper < 0)))\n    return dict(lift_by_fraction=by_fraction, unit="patient_cluster", population="patient_snapshots",\n                patients=len(unique), requested_draws=n_bootstrap, valid_draws=len(draws), single_class_draws=invalid,\n                seed=seed, confidence=confidence, direction="model_a_minus_model_b", paired=True,\n                interval_reliable=reliable, tie_rule="frozen original row order; repeated rows counted with patient multiplicity",\n                capacity_rule="exact ceil(fraction * resampled_snapshot_count)")\n\n\ndef compare_run(data, prep, protocol, results, lock, run_prefix, *, evaluate_test=False, n_bootstrap=1000):\n    if lock["protocol_hash"] != digest(protocol) or lock["input_manifest_hash"] != digest(data["manifest"]) or lock["preparation_hash"] != digest(prep):\n        raise ValueError("Evaluation lock mismatch")\n    saved = json.loads(read_artifacts(run_prefix + "_LOCK", {"lock.json"})["lock.json"])\n    if saved != plain(lock):\n        raise ValueError("Persisted lock mismatch")\n    split = "test" if evaluate_test else "validation"\n    table = run_prefix + ("_FINAL_TEST" if evaluate_test else "_VALIDATION_COMPARISON")\n    names = {"report.json"} | {n + ".npy" for n in protocol["arms"]}\n    if table_exists(table):\n        report = json.loads(read_artifacts(table, names)["report.json"])\n        if report["lock_hash"] != digest(lock) or report["bootstrap_resamples"] != n_bootstrap:\n            raise ValueError("Existing comparison differs from lock or uncertainty protocol")\n        return report\n    idx = data["indices"][split]\n    scores, metrics = {}, {}\n    for name, arm in protocol["arms"].items():\n        runs = results[name]\n        if [r["payload_hashes"] for r in runs] != lock["models"][name]["predictor_hashes"]:\n            raise ValueError("Final fitted model changed")\n        if evaluate_test:\n            x, valid, static = arm_arrays(data, prep, arm, "test")\n            try:\n                if arm["family"] != "transformer":\n                    raise ValueError("Only Transformer arms are registered; LightGBM remains deferred")\n                predict = modeling.predict_encoder\n                scores[name] = np.mean([predict(r, x, valid=valid, static=static) for r in runs], axis=0)\n            finally:\n                modeling.release_array(x)\n                modeling.release_array(static)\n        else:\n            scores[name] = np.mean([r["val_scores"] for r in runs], axis=0)\n        metrics[name] = comparison.binary_metrics(data["y"][idx], scores[name], lock["models"][name]["threshold"],\n                                                   top_fractions=protocol["top_fractions"])\n        metrics[name]["primary_lift"] = modeling.lift_at_fraction(data["y"][idx], scores[name], protocol["primary_lift_fraction"])\n        metrics[name]["primary_lift_fraction"] = protocol["primary_lift_fraction"]\n    pairs = [("SEQUENCE", "SUMMARY")] if "SEQUENCE" in scores and "SUMMARY" in scores else []\n    if "COUNT" in scores:\n        pairs += [(name, "COUNT") for name in ("SUMMARY", "SEQUENCE", "STATIC_FUSION", "TEMPORAL_FUSION", "NLP_FUSION") if name in scores]\n    if "STATIC_FUSION" in scores and "TEMPORAL_FUSION" in scores:\n        pairs += [("TEMPORAL_FUSION", "STATIC_FUSION")]\n    if "STATIC_FUSION" in scores and "NLP_FUSION" in scores:\n        pairs += [("NLP_FUSION", "STATIC_FUSION")]\n    for name in ("SUMMARY", "SEQUENCE"):\n        if "GB_" + name in scores:\n            pairs.append((name, "GB_" + name))\n    comparisons = {}\n    for a, b in pairs:\n        uncertainty = comparison.paired_patient_bootstrap(data["y"][idx], scores[a], scores[b],\n            data["metadata"].PATIENT_ID.to_numpy()[idx], n_bootstrap=n_bootstrap, seed=20261013)\n        lift_uncertainty = paired_lift_bootstrap(data["y"][idx], scores[a], scores[b],\n            data["metadata"].PATIENT_ID.to_numpy()[idx], fractions=protocol["top_fractions"],\n            n_bootstrap=n_bootstrap, seed=20261013)\n        comparisons[a + "_MINUS_" + b] = dict(model_a=a, model_b=b, **uncertainty,\n            primary_lift=lift_uncertainty["lift_by_fraction"][str(protocol["primary_lift_fraction"])],\n            lift_by_fraction=lift_uncertainty["lift_by_fraction"], lift_bootstrap=lift_uncertainty)\n    report = dict(lock_hash=digest(lock), split=split, metrics=metrics, comparisons=comparisons,\n                  test_prediction_performed=bool(evaluate_test),\n                  bootstrap_resamples=n_bootstrap, execution_status="completed_on_configured_inputs",\n                  primary_metric=protocol["primary_metric"], primary_lift_fraction=protocol["primary_lift_fraction"],\n                  lift_population=protocol["lift_population"], lift_tie_policy=protocol["lift_tie_policy"],\n                  source_id=data["manifest"]["panel_manifest"]["contract"]["source_id"],\n                  test_evaluation_ordinal=protocol["test_evaluation_ordinal"] if evaluate_test else None,\n                  limitations=["Validation is development evidence, not independent confirmation" if not evaluate_test else protocol["test_scope"],\n                    "Bootstrap conditions on fitted predictions; excludes training/search uncertainty",\n                    "Validation intervals condition on validation-selected checkpoints and are optimistic development evidence",\n                    "Cohort and source-vintage comparability must be verified before comparing absolute lift with prior runs",\n                    "Unadjusted intervals across fixed contrasts; no multiplicity-controlled superiority claim",\n                    "Augmentation changes input width/capacity as reported; no matched zero-input capacity ablation",\n                    "Primary capacity was fixed before fitting; other capacity points are secondary, not selected from TEST",\n                    "Lift counts patient snapshots; repeated monthly opportunities are not unique-patient intervention counts",\n                    "Input cutoff and source availability claims limited to recorded construction evidence",\n                    "Augmentation comparisons are conditional on the registered training/selection procedure; no causal claim"],\n                  count_reference_status="matched refit provided" if "COUNT" in scores else "not available; added-value-versus-1028 conclusion pending",\n                  lightgbm_status="included" if "GB_SUMMARY" in scores else "deferred")\n    save_artifacts(table, dict({"report.json": json_bytes(report)}, **{n + ".npy": array_bytes(s) for n, s in scores.items()}))\n    return report\n\n\ndef report_table(report):\n    return pd.DataFrame([dict(arm=name, split=report["split"],\n        **{"lift_" + str(int(round(100*row["fraction"]))) + "pct": row["lift"] for row in values["topk"]},\n        **{key: values.get(key) for key in\n        ("primary_lift", "primary_lift_fraction", "snapshots", "positives", "average_precision", "roc_auc", "log_loss", "brier_score", "precision", "recall", "f1", "specificity")})\n        for name, values in report["metrics"].items()])\n', 'comparison': '"""Aggregate evaluation for locked finalists, never model selection on TEST.\n\nAverage precision (AP) and trapezoidal PR area are deliberately separate.\nBootstrap intervals condition on the fitted predictions and resample patients,\nkeeping all snapshots and both models paired. They do not include retraining\nvariation, resolve unknown provenance, or make a reused holdout prospective.\n"""\nimport hashlib\nimport json\nimport math\n\nimport numpy as np\nfrom sklearn.metrics import (average_precision_score, brier_score_loss, log_loss,\n                             precision_recall_curve, roc_auc_score)\n\n\ndef _inputs(y, scores):\n    y, scores = np.asarray(y), np.asarray(scores, dtype=float)\n    if y.ndim != 1 or scores.ndim != 1 or not len(y) or len(y) != len(scores):\n        raise ValueError("Labels and probability scores must be equal nonempty vectors")\n    if not np.isin(y, [0, 1]).all():\n        raise ValueError("Labels must be binary and nonmissing")\n    if not np.isfinite(scores).all() or ((scores < 0) | (scores > 1)).any():\n        raise ValueError("Scores must be finite probabilities in [0,1]; logits require an explicit conversion")\n    return y.astype(np.int8), scores\n\n\ndef _ratio(numerator, denominator):\n    return float(numerator / denominator) if denominator else None\n\n\ndef _json_hash(value):\n    return hashlib.sha256(json.dumps(value, sort_keys=True, separators=(",", ":"),\n                                     allow_nan=False).encode()).hexdigest()\n\n\ndef _tie_keys(n, tie_breaker):\n    if tie_breaker is None:\n        return np.arange(n), "frozen_input_row_order"\n    keys = np.asarray(tie_breaker).astype(str)\n    if keys.ndim != 1 or len(keys) != n or len(np.unique(keys)) != n:\n        raise ValueError("Tie keys must be unique, stable snapshot keys in prediction order")\n    return keys, "supplied_unique_snapshot_keys"\n\n\ndef topk_metrics(y, scores, fraction=0.1, tie_breaker=None):\n    """Exactly ceil(fraction*N) selected; stable ties are independent of labels."""\n    y, scores = _inputs(y, scores)\n    if not 0 < float(fraction) <= 1:\n        raise ValueError("Top fraction must lie in (0,1]")\n    keys, tie_rule = _tie_keys(len(y), tie_breaker)\n    k = max(1, int(math.ceil(float(fraction) * len(y))))\n    order = np.lexsort((keys, -scores))\n    selected = order[:k]\n    positives = int(y.sum())\n    captured = int(y[selected].sum())\n    boundary = float(scores[selected[-1]])\n    tied = scores == boundary\n    tied_selected = int(tied[selected].sum())\n    precision = captured / k\n    prevalence = positives / len(y)\n    return dict(fraction=float(fraction), selected=k, selected_positives=captured,\n                actual_population_fraction=k / len(y), precision=precision,\n                recall=_ratio(captured, positives), lift=_ratio(precision, prevalence),\n                cutoff_score=boundary, n_tied_at_cutoff=int(tied.sum()),\n                n_tied_selected=tied_selected, tie_crosses_boundary=bool(tied.sum() > tied_selected),\n                tie_rule=tie_rule, rule="ceil(fraction*N), descending probability, ascending stable key")\n\n\ndef binary_metrics(y, scores, threshold=None, top_fractions=(0.1,), calibration_bins=10,\n                   tie_breaker=None):\n    y, scores = _inputs(y, scores)\n    if isinstance(calibration_bins, bool) or not 1 <= int(calibration_bins) <= 100:\n        raise ValueError("Use 1 to 100 fixed-width calibration bins")\n    calibration_bins = int(calibration_bins)\n    positives, n = int(y.sum()), len(y)\n    both = 0 < positives < n\n    precision, recall, _ = precision_recall_curve(y, scores) if positives else (None, None, None)\n    integrate = getattr(np, "trapezoid", np.trapz if hasattr(np, "trapz") else None)\n    result = dict(snapshots=n, positives=positives, negatives=n - positives, prevalence=positives / n,\n                  average_precision=float(average_precision_score(y, scores)) if positives else None,\n                  pr_auc_trapezoidal=float(integrate(precision[::-1], recall[::-1])) if positives else None,\n                  roc_auc=float(roc_auc_score(y, scores)) if both else None,\n                  brier_score=float(brier_score_loss(y, scores)),\n                  log_loss=float(log_loss(y, scores, labels=[0, 1])),\n                  metric_definition="AP=sum(recall increment*precision); PR area uses trapezoids; these are not interchangeable",\n                  single_class=not both)\n    if threshold is not None:\n        if isinstance(threshold, dict):\n            if threshold.get("selection_split") != "validation" or threshold.get("objective") != "f1":\n                raise ValueError("Threshold policy must be locked on VALIDATION")\n            if threshold.get("policy_sha256") != _json_hash({k: v for k, v in threshold.items() if k != "policy_sha256"}):\n                raise ValueError("Validation threshold policy hash mismatch")\n            value = float(threshold["threshold"])\n        else:\n            value = float(threshold)\n        if not np.isfinite(value) or not 0 <= value <= 1:\n            raise ValueError("Probability threshold must lie in [0,1]")\n        predicted = scores >= value\n        tp, fp = int((predicted & (y == 1)).sum()), int((predicted & (y == 0)).sum())\n        fn, tn = positives - tp, n - positives - fp\n        result.update(threshold=value, tp=tp, fp=fp, fn=fn, tn=tn,\n                      precision=_ratio(tp, tp + fp), recall=_ratio(tp, tp + fn),\n                      specificity=_ratio(tn, tn + fp), f1=_ratio(2 * tp, 2 * tp + fp + fn),\n                      accuracy=(tp + tn) / n,\n                      confusion_matrix=[[tn, fp], [fn, tp]],\n                      threshold_scope="validation_locked" if isinstance(threshold, dict) else "caller_supplied")\n    bins = np.minimum((scores * calibration_bins).astype(int), calibration_bins - 1)\n    rows = []\n    for b in range(calibration_bins):\n        mask = bins == b\n        count = int(mask.sum())\n        rows.append(dict(bin=b, lower=b / calibration_bins, upper=(b + 1) / calibration_bins,\n                         count=count, mean_probability=float(scores[mask].mean()) if count else None,\n                         observed_rate=float(y[mask].mean()) if count else None))\n    result["calibration"] = rows\n    result["expected_calibration_error"] = float(sum(\n        row["count"] / n * abs(row["mean_probability"] - row["observed_rate"])\n        for row in rows if row["count"]))\n    result["calibration_note"] = "Fixed-width descriptive calibration; no calibration model fitted to evaluation labels"\n    result["topk"] = [topk_metrics(y, scores, f, tie_breaker) for f in top_fractions]\n    return result\n\n\ndef select_validation_threshold(y_validation, scores_validation, objective="f1", *, selection_split="validation"):\n    """Choose once on VALIDATION; highest threshold breaks an exact F1 tie."""\n    if selection_split != "validation" or objective != "f1":\n        raise ValueError("Only the prespecified VALIDATION F1 policy is supported")\n    y, scores = _inputs(y_validation, scores_validation)\n    if len(np.unique(y)) != 2:\n        raise ValueError("Threshold selection requires both classes in VALIDATION")\n    precision, recall, thresholds = precision_recall_curve(y, scores)\n    p, r = precision[:-1], recall[:-1]\n    f1 = np.divide(2 * p * r, p + r, out=np.zeros_like(p), where=(p + r) > 0)\n    best = np.flatnonzero(f1 == f1.max())[-1]\n    record = dict(threshold=float(thresholds[best]), objective=objective, selection_split="validation",\n                  validation_f1=float(f1[best]), validation_snapshots=len(y),\n                  tie_rule="highest probability threshold among exactly equal maximum F1 values",\n                  decision_rule="score >= threshold")\n    record["policy_sha256"] = _json_hash(record)\n    return record\n\n\ndef lock_validation_policy(y_validation, scores_validation, top_fractions=(0.1,)):\n    """Top-K locks population fractions, not a score cutoff derived from TEST."""\n    fractions = list(map(float, top_fractions))\n    if any(not 0 < f <= 1 for f in fractions):\n        raise ValueError("Invalid operational population fractions")\n    return dict(f1=select_validation_threshold(y_validation, scores_validation),\n                top_fractions=fractions,\n                topk_rule="ceil(fraction*N); descending score; ascending prespecified unique snapshot key",\n                selection_split="validation")\n\n\ndef _patient_codes(patient_ids, n):\n    patients = np.asarray(patient_ids)\n    if patients.ndim != 1 or len(patients) != n:\n        raise ValueError("Patient groups must match predictions")\n    if any(v is None or str(v).strip().lower() in ("", "nan", "none", "nat") for v in patients):\n        raise ValueError("Patient IDs must be nonmissing")\n    _, codes = np.unique(patients.astype(str), return_inverse=True)\n    return codes\n\n\ndef paired_patient_bootstrap(y, scores_a, scores_b, patient_ids, n_bootstrap=1000,\n                             seed=20261006, confidence=0.95):\n    """Delta = model A minus B. Cluster multiplicities act as row weights."""\n    y, a = _inputs(y, scores_a)\n    _, b = _inputs(y, scores_b)\n    if len(np.unique(y)) != 2:\n        raise ValueError("Paired AP/AUC inference requires both classes")\n    if isinstance(n_bootstrap, bool) or int(n_bootstrap) != n_bootstrap or n_bootstrap < 20:\n        raise ValueError("At least 20 bootstrap draws required; use >=1000 for reports")\n    if not 0 < confidence < 1:\n        raise ValueError("Confidence must lie in (0,1)")\n    codes = _patient_codes(patient_ids, len(y))\n    groups = int(codes.max()) + 1\n    if groups < 2:\n        raise ValueError("At least two patients required")\n    rng = np.random.default_rng(seed)\n    draws, invalid = [], 0\n    for _ in range(int(n_bootstrap)):\n        multiplicity = np.bincount(rng.integers(0, groups, size=groups), minlength=groups)\n        weights = multiplicity[codes]\n        if weights[y == 1].sum() == 0 or weights[y == 0].sum() == 0:\n            invalid += 1\n            continue\n        ap = average_precision_score(y, a, sample_weight=weights) - average_precision_score(y, b, sample_weight=weights)\n        auc = roc_auc_score(y, a, sample_weight=weights) - roc_auc_score(y, b, sample_weight=weights)\n        draws.append((ap, auc))\n    result = dict(unit="patient", patients=groups, requested_draws=int(n_bootstrap), valid_draws=len(draws),\n                  single_class_draws=invalid, seed=int(seed), confidence=float(confidence),\n                  direction="model_a_minus_model_b", paired=True,\n                  uncertainty_scope="conditional on fitted predictions; excludes retraining and model-selection uncertainty")\n    enough = len(draws) >= max(20, int(math.ceil(0.8 * n_bootstrap)))\n    result["interval_reliable"] = enough\n    values = np.asarray(draws) if draws else np.empty((0, 2))\n    alpha = (1 - confidence) / 2\n    for index, (name, func) in enumerate((("average_precision", average_precision_score), ("roc_auc", roc_auc_score))):\n        result[name] = dict(delta=float(func(y, a) - func(y, b)),\n                            lower=float(np.quantile(values[:, index], alpha)) if enough else None,\n                            upper=float(np.quantile(values[:, index], 1 - alpha)) if enough else None)\n    return result\n\n\ndef latest_patient_indices(patient_ids, end_dates):\n    """One latest snapshot per patient; reject ambiguous duplicate latest keys."""\n    dates = np.asarray(end_dates, dtype="datetime64[D]")\n    codes = _patient_codes(patient_ids, len(dates))\n    if dates.ndim != 1 or np.isnat(dates).any():\n        raise ValueError("Latest-patient evaluation requires exact nonmissing dates")\n    result = []\n    for patient in range(int(codes.max()) + 1):\n        rows = np.flatnonzero(codes == patient)\n        latest = rows[dates[rows] == dates[rows].max()]\n        if len(latest) != 1:\n            raise ValueError("Duplicate latest patient-snapshot keys")\n        result.append(int(latest[0]))\n    return np.asarray(result, dtype=int)\n\n\ndef summarize_seed_runs(results):\n    """Accept training.py\'s candidate -> complete seed-run list; retain every seed."""\n    if not isinstance(results, dict) or not results:\n        raise ValueError("Expected candidate-to-seed-runs dictionary")\n    all_seeds, candidates = [], []\n    for candidate, runs in results.items():\n        seeds = []\n        for run in runs:\n            summary = run["summary"]\n            if not summary.get("training_complete"):\n                raise ValueError("Incomplete runs cannot disappear from the final comparison")\n            seed = int(summary["seed"])\n            if seed in seeds:\n                raise ValueError("Duplicate seed in candidate")\n            seeds.append(seed)\n            row = dict(candidate=candidate, seed=seed,\n                       family=summary["config"]["family"],\n                       parameter_count=summary.get("parameter_count"),\n                       complexity_unit=summary.get("complexity_unit"),\n                       best_epoch=summary.get("best_epoch"), wall_seconds=summary.get("wall_seconds"),\n                       validation_evaluations=summary.get("validation_evaluations"))\n            for split in ("train", "validation", "test"):\n                for metric, value in summary.get(split + "_metrics", {}).items():\n                    if isinstance(value, (int, float, np.number)) and not isinstance(value, (bool, np.bool_)):\n                        row[split + "_" + metric] = float(value)\n            train_ap, val_ap = row.get("train_average_precision"), row.get("validation_average_precision")\n            row["train_validation_gap_ap"] = train_ap - val_ap if train_ap is not None and val_ap is not None else None\n            test_ap = row.get("test_average_precision")\n            row["train_test_gap_ap"] = train_ap - test_ap if train_ap is not None and test_ap is not None else None\n            all_seeds.append(row)\n        if not seeds:\n            raise ValueError("Candidate has no completed seed repetitions")\n        candidate_rows = [r for r in all_seeds if r["candidate"] == candidate]\n        values = [r["validation_average_precision"] for r in candidate_rows]\n        candidates.append(dict(candidate=candidate, seeds=seeds, n_seeds=len(seeds),\n                               mean_validation_ap=float(np.mean(values)),\n                               sd_validation_ap=float(np.std(values, ddof=1)) if len(values) > 1 else None,\n                               min_validation_ap=float(np.min(values)), max_validation_ap=float(np.max(values)),\n                               seed_selection="all prespecified seeds, none selected by outcome"))\n    return dict(all_seeds=all_seeds, candidates=candidates,\n                note="Seed SD is training variability, not a confidence interval; bootstrap intervals are separate")\n\n\ndef comparison_conclusion(bootstrap, provenance):\n    """Wording is gated by comparison provenance, never VALIDATION significance."""\n    provenance = dict(provenance or {})\n    base = dict(status="UNRESOLVED", conclusion=None, restrictions=[])\n    if provenance.get("baseline_kind") == "original_v63" or not provenance.get("fit_membership_verified", False):\n        base.update(status="DESCRIPTIVE_ONLY", conclusion="The existing LightGBM scores have unresolved fitting provenance; they cannot establish comparative generalization or Transformer superiority.")\n        return base\n    required = ("same_cohort", "same_labels", "patient_disjoint_split", "selection_locked")\n    missing = [key for key in required if not provenance.get(key, False)]\n    if missing or provenance.get("test_used_for_selection", True):\n        base["restrictions"] = missing + (["test_used_for_selection_or_unknown"] if provenance.get("test_used_for_selection", True) else [])\n        base["conclusion"] = "A matched final comparison is not established; resolve the listed protocol conditions before making a superiority claim."\n        return base\n    if provenance.get("comparison_split") != "test":\n        base.update(status="DEVELOPMENT_ONLY", conclusion="Validation differences are development evidence after model selection; final-test superiority is not established.")\n        return base\n    for key in ("upstream_cutoff_verified", "holdout_never_previously_inspected"):\n        if not provenance.get(key, False):\n            base["restrictions"].append(key)\n    if not provenance.get("same_feature_information", False):\n        base["restrictions"].append("different_feature_information_architecture_effect_not_isolated")\n    ap = bootstrap["average_precision"]\n    if not bootstrap.get("interval_reliable") or ap["lower"] is None:\n        base.update(status="UNCERTAINTY_UNRESOLVED", conclusion="Too few valid patient-bootstrap draws support a reliable interval; report the point difference without a superiority conclusion.")\n        return base\n    prefix = "In this retrospective matched comparison" if base["restrictions"] else "On the locked patient-disjoint test cohort"\n    if ap["lower"] > 0:\n        verdict = "Transformer AP is higher than matched LightGBM AP"\n        status = "RETROSPECTIVE_AP_ADVANTAGE" if base["restrictions"] else "TEST_AP_ADVANTAGE"\n    elif ap["upper"] < 0:\n        verdict = "matched LightGBM AP is higher than Transformer AP"\n        status = "LIGHTGBM_AP_ADVANTAGE"\n    else:\n        verdict = "the AP interval includes no difference; superiority is not established and this is not proof of equivalence"\n        status = "NO_DEMONSTRATED_AP_ADVANTAGE"\n    base.update(status=status, conclusion=f"{prefix}, {verdict}. Delta AP (Transformer minus LightGBM) = {ap[\'delta\']:.6f}, patient-bootstrap interval [{ap[\'lower\']:.6f}, {ap[\'upper\']:.6f}]. This interval conditions on the fitted models and excludes training/search uncertainty.")\n    if base["restrictions"]:\n        base["conclusion"] += " Unresolved provenance or prior holdout inspection prevents a prospective generalization claim."\n    return base\n\n\ndef compare_predictions(y, transformer_scores, lightgbm_scores, patient_ids, *,\n                        threshold_policies=None, provenance=None, tie_breaker=None,\n                        top_fractions=(0.1,), n_bootstrap=1000, seed=20261006):\n    """Caller aligns both score vectors to identical frozen keys and persists lock first."""\n    policies = threshold_policies or {}\n    t = binary_metrics(y, transformer_scores, policies.get("transformer"), top_fractions=top_fractions,\n                       tie_breaker=tie_breaker)\n    l = binary_metrics(y, lightgbm_scores, policies.get("lightgbm"), top_fractions=top_fractions,\n                       tie_breaker=tie_breaker)\n    interval = paired_patient_bootstrap(y, transformer_scores, lightgbm_scores, patient_ids,\n                                        n_bootstrap=n_bootstrap, seed=seed)\n    return dict(transformer=t, lightgbm=l, paired_uncertainty=interval,\n                conclusion=comparison_conclusion(interval, provenance), provenance=dict(provenance or {}))\n', 'warehouse': '"""Embedded by the delivery notebooks; no repository dependency at runtime."""\nimport base64\nimport hashlib\nimport io\nimport json\nimport math\nimport re\nimport tempfile\nimport uuid\nfrom contextlib import contextmanager\nfrom itertools import islice\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\n\n\ndef canonical_json(value):\n    return json.dumps(value, ensure_ascii=False, sort_keys=True, separators=(",", ":"), allow_nan=False)\n\n\ndef digest_json(value):\n    return hashlib.sha256(canonical_json(value).encode("utf-8")).hexdigest()\n\n\ndef read_sf(suffix):\n    return (spark.read.format("snowflake").options(**sf_options_dl_poc)\n            .option("dbtable", f"{PREFIX}_{suffix}").load())\n\n\ndef table_exists(table):\n    if not re.fullmatch(r"[A-Z][A-Z0-9_]*", table):\n        raise ValueError("Use uppercase letters, numbers and underscores in table names.")\n    query = ("SELECT TABLE_NAME FROM DSVC_TAKEDA_TA_PRIVATE.INFORMATION_SCHEMA.TABLES "\n             f"WHERE TABLE_SCHEMA = \'DS_ML\' AND TABLE_NAME = \'{table}\'")\n    return (spark.read.format("snowflake").options(**sf_options_dl_poc)\n            .option("query", query).load().limit(1).count() > 0)\n\n\ndef checked_metadata(frame, include_split=False):\n    columns = ["PATIENT_ID", "END_DT", "RESP"]\n    if include_split:\n        columns += ["SPLIT", "SPLIT_CONFIG"]\n    out = frame.loc[:, columns].copy()\n    if out.empty or out.isna().any().any():\n        raise ValueError("Snapshot metadata must be nonempty and contain no nulls.")\n    if not out.PATIENT_ID.map(lambda x: isinstance(x, str) and bool(x)).all():\n        raise ValueError("PATIENT_ID must retain its original nonempty string value.")\n    dates = pd.to_datetime(out.END_DT, errors="raise")\n    if dates.dt.tz is not None or not dates.eq(dates.dt.normalize()).all():\n        raise ValueError("END_DT must be a date without an intraday time/timezone.")\n    out["END_DT"] = dates.dt.strftime("%Y-%m-%d")\n    if not out.RESP.isin([0, 1]).all():\n        raise ValueError("RESP must be exactly 0 or 1 before conversion.")\n    out["RESP"] = out.RESP.astype("int64")\n    if out.duplicated(["PATIENT_ID", "END_DT"]).any():\n        raise ValueError("Duplicate patient/date keys in snapshot metadata.")\n    if include_split:\n        if set(out.SPLIT) != {"train", "validation", "test"}:\n            raise ValueError("Expected the saved train, validation and test assignments.")\n        if out.groupby("PATIENT_ID", observed=True).SPLIT.nunique().gt(1).any():\n            raise ValueError("Patient overlap between splits.")\n        if len(out.SPLIT_CONFIG.unique()) != 1:\n            raise ValueError("The split contains inconsistent creation settings.")\n        for _, part in out.groupby("SPLIT", observed=True):\n            if set(part.RESP) != {0, 1}:\n                raise ValueError("Each split must contain both response classes.")\n    return out.sort_values(["PATIENT_ID", "END_DT"]).reset_index(drop=True)\n\n\ndef validate_metadata_pair(snapshot_frame, manifest_frame, features):\n    source = checked_metadata(snapshot_frame)\n    manifest = checked_metadata(manifest_frame, include_split=True)\n    if not source.equals(manifest[["PATIENT_ID", "END_DT", "RESP"]]):\n        raise ValueError("Frozen split and source snapshots differ in keys or labels.")\n    creation = json.loads(manifest.SPLIT_CONFIG.iloc[0])\n    \n    feature_order_hash = hashlib.sha256(\n        json.dumps(features, ensure_ascii=False).encode("utf-8")).hexdigest()\n    if creation.get("feature_order_sha256") != feature_order_hash or creation.get("n_timesteps") != 12:\n        raise ValueError("Feature order or timesteps differ from the frozen split.")\n    return manifest, creation\n\n\ndef new_tensor_buffer(metadata, feature_count, seq_len=12):\n    temporary = tempfile.TemporaryDirectory(prefix="dl_poc_")\n    path = Path(temporary.name) / "counts.float32"\n    X = np.memmap(path, dtype="<f4", mode="w+", shape=(len(metadata), seq_len, feature_count))\n    return temporary, X\n\n\ndef fill_tensor(X, metadata, sequence_rows):\n    """Place rows by canonical keys, independent of Spark partition order."""\n    positions = {(r.PATIENT_ID, r.END_DT): i for i, r in enumerate(metadata.itertuples())}\n    seen = np.zeros(len(metadata), dtype=bool)\n    seq_len, feature_count = X.shape[1:]\n    for row in sequence_rows:\n        raw_date = row["END_DT"]\n        if raw_date is None or row["PATIENT_ID"] is None:\n            raise ValueError("Null monthly snapshot key.")\n        date = pd.Timestamp(raw_date)\n        if date.tzinfo is not None or date != date.normalize():\n            raise ValueError("Monthly END_DT is not an exact date.")\n        key = (row["PATIENT_ID"], date.strftime("%Y-%m-%d"))\n        if key not in positions:\n            raise ValueError("Unexpected monthly snapshot key.")\n        i = positions[key]\n        if seen[i] or row["RESP"] != int(metadata.RESP.iloc[i]):\n            raise ValueError("Duplicate monthly snapshot or changed label.")\n        sequence = row["SEQUENCE"]\n        steps = [month["T"] for month in sequence]\n        if len(sequence) != seq_len or any(t is None for t in steps):\n            raise ValueError("Expected exactly 12 complete timesteps per snapshot.")\n        \n        if sorted(steps) != list(range(seq_len)):\n            raise ValueError("Timesteps must be unique integers 0 through 11.")\n        sequence = sorted(sequence, key=lambda month: month["T"])\n        values = np.asarray([month["V"] for month in sequence], dtype=np.float32)\n        if values.shape != (seq_len, feature_count):\n            raise ValueError("Monthly feature shape does not match the vocabulary.")\n        if not np.isfinite(values).all() or (values < 0).any():\n            raise ValueError("Counts must be finite, nonnegative float32 values with no nulls.")\n        X[i] = values\n        seen[i] = True\n    if not seen.all():\n        raise ValueError("Monthly data is missing original snapshots, including zero-activity sequences.")\n    X.flush()\n\n\ndef input_fingerprints(X, metadata, features):\n    tensor_hash = hashlib.sha256()\n    tensor_hash.update(canonical_json(list(X.shape)).encode("utf-8"))\n    for start in range(0, len(X), 128):\n        tensor_hash.update(np.asarray(X[start:start + 128], dtype="<f4").tobytes(order="C"))\n    records = [[str(r.PATIENT_ID), str(r.END_DT), int(r.RESP), str(r.SPLIT)]\n               for r in metadata.itertuples()]\n    return {"model_input_sha256": tensor_hash.hexdigest(),\n            "snapshot_manifest_sha256": digest_json(records),\n            "feature_names_sha256": digest_json(features),\n            "split_config_sha256": digest_json(json.loads(metadata.SPLIT_CONFIG.iloc[0]))}\n\n\ndef load_inputs():\n    from pyspark.sql import functions as F\n    mapping = read_sf("FEATURE_MAP").orderBy("FEATURE_INDEX").collect()\n    if not mapping or [r["FEATURE_INDEX"] for r in mapping] != list(range(len(mapping))):\n        raise ValueError("Invalid feature indices.")\n    features = [r["FEATURE_NAME"] for r in mapping]\n    aliases = [r["FEATURE_COLUMN"] for r in mapping]\n    if (aliases != [f"F{i:04d}" for i in range(len(features))]\n            or not all(isinstance(f, str) and f for f in features)\n            or len(set(features)) != len(features)):\n        raise ValueError("Invalid feature names, aliases or order.")\n    source = read_sf("SNAPSHOTS").select("PATIENT_ID", "END_DT", "RESP").toPandas()\n    frozen = read_sf("PATIENT_SPLIT").select(\n        "PATIENT_ID", "END_DT", "RESP", "SPLIT", "SPLIT_CONFIG").toPandas()\n    metadata, creation = validate_metadata_pair(source, frozen, features)\n    observed = (len(metadata), metadata.PATIENT_ID.nunique(), int(metadata.RESP.sum()), len(features))\n    if observed != (23151, 12447, 1345, 1028):\n        raise ValueError(f"V63 population changed: snapshots/patients/positives/features = {observed}.")\n    monthly = read_sf("TENSOR_MONTHLY")\n    if set(monthly.columns) != set(["PATIENT_ID", "END_DT", "RESP", "TIME_STEP"] + aliases):\n        raise ValueError("Monthly table columns differ from the frozen feature map.")\n    print("Building the model input in a temporary driver file (about 1.06 GiB).", flush=True)\n    grouped = (monthly.groupBy("PATIENT_ID", "END_DT", "RESP")\n        .agg(F.collect_list(F.struct(\n            F.col("TIME_STEP").alias("T"),\n            F.array(*[F.when(F.col(name) >= 0, F.col(name).cast("float"))\n                      .otherwise(F.lit(None).cast("float")) for name in aliases]).alias("V")\n        )).alias("SEQUENCE"))\n        .repartition(128))\n    temporary, X = new_tensor_buffer(metadata, len(features))\n    try:\n        fill_tensor(X, metadata, grouped.toLocalIterator(prefetchPartitions=False))\n        hashes = input_fingerprints(X, metadata, features)\n        X.flags.writeable = False\n    except BaseException:\n        X._mmap.close()\n        temporary.cleanup()\n        raise\n    y = metadata.RESP.to_numpy(dtype=np.float32)\n    indices = {name: np.flatnonzero(metadata.SPLIT.to_numpy() == name)\n               for name in ("train", "validation", "test")}\n    print(f"Validated tensor shape {X.shape}; patient overlap = 0.", flush=True)\n    return {"X": X, "y": y, "metadata": metadata, "features": features,\n            "indices": indices, "hashes": hashes, "split_creation": creation,\n            "temporary_directory": temporary}\n\n\n@contextmanager\ndef _blob_reader(blob):\n    """Bytes and caller-owned files share the same raw-byte hash contract."""\n    if isinstance(blob, Path):\n        with blob.open("rb") as stream:\n            yield stream\n    elif isinstance(blob, (bytes, bytearray, memoryview)):\n        \n        class Reader:\n            offset = 0\n            def read(self, size):\n                block = memoryview(blob)[self.offset:self.offset + size]\n                self.offset += len(block)\n                return block\n        yield Reader()\n    else:\n        raise TypeError("Artifact payload must be bytes or pathlib.Path")\n\n\ndef artifact_signature(blob):\n    hasher, size = hashlib.sha256(), 0\n    with _blob_reader(blob) as stream:\n        while True:\n            block = stream.read(1024 * 1024)\n            if not block:\n                break\n            hasher.update(block)\n            size += len(block)\n    return size, hasher.hexdigest()\n\n\ndef iter_artifact_rows(artifacts, chunk_size=50000, signatures=None):\n    """Same rows as the former packer, without whole-blob/base64/list copies."""\n    if not isinstance(chunk_size, int) or chunk_size < 1:\n        raise ValueError("chunk_size must be a positive integer")\n    for name, blob in artifacts.items():\n        size, digest = signatures[name] if signatures is not None else artifact_signature(blob)\n        count = max(1, math.ceil((4 * ((size + 2) // 3)) / chunk_size))\n        \n        raw_size = max(3, (chunk_size // 4) * 3)\n        pending, index = "", 0\n        with _blob_reader(blob) as stream:\n            while True:\n                block = stream.read(raw_size)\n                if not block:\n                    break\n                pending += base64.b64encode(block).decode("ascii")\n                while len(pending) >= chunk_size:\n                    yield (name, index, count, size, digest, pending[:chunk_size])\n                    pending = pending[chunk_size:]\n                    index += 1\n        if pending or size == 0:\n            yield (name, index, count, size, digest, pending)\n\n\ndef pack_artifacts(artifacts, chunk_size=50000):\n    """Compatibility helper for small callers; production writes use the iterator."""\n    return list(iter_artifact_rows(artifacts, chunk_size))\n\n\ndef unpack_artifacts(rows, expected_names, *, disk_names=(), output_dir=None, ordered=False):\n    """Decode ordered chunks incrementally; optional large payloads remain on disk.\n\n    The default still accepts unordered small row collections. read_artifacts\n    supplies bounded, ordered chunks, so it never takes that sorting path.\n    """\n    names, disk_names = set(expected_names), set(disk_names)\n    if not disk_names <= names or (disk_names and output_dir is None):\n        raise ValueError("disk_names must be expected names and require output_dir")\n    if not ordered:\n        rows = sorted(rows, key=lambda row: (row[0], row[1]))\n    result, created, stream = {}, [], None\n    current, count, size, digest, next_index, actual_size, pending = None, 0, 0, None, 0, 0, ""\n\n    def finish():\n        nonlocal stream\n        if next_index != count or pending or actual_size != size or hasher.hexdigest() != digest:\n            raise ValueError("Missing, duplicate or inconsistent artifact chunks, or length/hash mismatch.")\n        if current in disk_names:\n            stream.close()\n            result[current] = path\n        else:\n            result[current] = stream.getvalue()\n            stream.close()\n        stream = None\n\n    try:\n        for row in rows:\n            name, i, row_count, row_size, row_digest, payload = tuple(row)\n            if any(value != int(value) for value in (i, row_count, row_size)):\n                raise ValueError("Nonintegral artifact chunk metadata.")\n            i, row_count, row_size = int(i), int(row_count), int(row_size)\n            if name != current:\n                if current is not None:\n                    finish()\n                if name not in names or name in result or row_count < 1 or row_size < 0:\n                    raise ValueError("Missing, duplicate or unexpected saved artifacts.")\n                current, count, size, digest = name, row_count, row_size, row_digest\n                next_index, actual_size, pending, hasher = 0, 0, "", hashlib.sha256()\n                if name in disk_names:\n                    directory = Path(output_dir)\n                    directory.mkdir(parents=True, exist_ok=True)\n                    \n                    path = directory / ("artifact_" + uuid.uuid4().hex)\n                    stream = path.open("xb")\n                    created.append(path)\n                else:\n                    stream = io.BytesIO()\n            if i != next_index or (row_count, row_size, row_digest) != (count, size, digest):\n                raise ValueError("Missing, duplicate or inconsistent artifact chunks.")\n            pending += payload\n            length = (len(pending) // 4) * 4\n            if length:\n                encoded, pending = pending[:length], pending[length:]\n                block = base64.b64decode(encoded, validate=True)\n                \n                if "=" in encoded and next_index != count - 1:\n                    raise ValueError("Unexpected base64 padding before final artifact chunk")\n                stream.write(block)\n                hasher.update(block)\n                actual_size += len(block)\n            next_index += 1\n        if current is not None:\n            finish()\n        if set(result) != names:\n            raise ValueError("Missing or unexpected saved artifacts.")\n        return result\n    except BaseException:\n        if stream is not None:\n            stream.close()\n        for path in created:\n            path.unlink(missing_ok=True)\n        raise\n\n\nARTIFACT_COLUMNS = ["ARTIFACT_NAME", "CHUNK_INDEX", "CHUNK_COUNT", "BYTE_LENGTH", "SHA256", "PAYLOAD_BASE64"]\nARTIFACT_BATCH_ROWS = 1024\n\n\ndef _checked_table(table):\n    if not isinstance(table, str) or not re.fullmatch(r"[A-Z][A-Z0-9_]*", table):\n        raise ValueError("Use uppercase letters, numbers and underscores in table names.")\n    return table\n\n\ndef _query_artifacts(query):\n    return (spark.read.format("snowflake").options(**sf_options_dl_poc)\n            .option("query", query).load())\n\n\ndef _read_artifact_rows(table, expected_names, batch_rows=ARTIFACT_BATCH_ROWS):\n    """Bound each connector result, not just the Python iterator\'s buffer."""\n    table = _checked_table(table)\n    if not isinstance(batch_rows, int) or batch_rows < 1:\n        raise ValueError("batch_rows must be positive")\n    metadata = _query_artifacts(\n        f"SELECT ARTIFACT_NAME,CHUNK_COUNT,BYTE_LENGTH,SHA256,COUNT(*) AS ROW_COUNT FROM {table} "\n        "GROUP BY ARTIFACT_NAME,CHUNK_COUNT,BYTE_LENGTH,SHA256").limit(len(set(expected_names)) + 1).collect()\n    descriptions = {}\n    for row in metadata:\n        name, count, size, digest, actual_count = tuple(row)\n        if (name in descriptions or name not in expected_names\n                or any(value != int(value) for value in (count, size, actual_count))\n                or count < 1 or size < 0 or count != actual_count):\n            raise ValueError("Missing, duplicate or inconsistent artifact chunks.")\n        descriptions[name] = (int(count), int(size), digest)\n    if set(descriptions) != set(expected_names):\n        raise ValueError("Missing or unexpected saved artifacts.")\n    for name in sorted(descriptions):\n        count = descriptions[name][0]\n        quoted_name = "\'" + str(name).replace("\'", "\'\'") + "\'"\n        for start in range(0, count, batch_rows):\n            query = ("SELECT " + ",".join(ARTIFACT_COLUMNS) + f" FROM {table} "\n                f"WHERE ARTIFACT_NAME={quoted_name} AND CHUNK_INDEX >= {start} "\n                f"AND CHUNK_INDEX < {min(start + batch_rows, count)} ORDER BY CHUNK_INDEX")\n            \n            batch = _query_artifacts(query).limit(batch_rows + 1).collect()\n            yield from sorted(batch, key=lambda row: row[1])\n            del batch\n\n\ndef read_artifacts(table, expected_names, *, disk_names=(), output_dir=None, batch_rows=ARTIFACT_BATCH_ROWS):\n    return unpack_artifacts(_read_artifact_rows(table, expected_names, batch_rows), expected_names,\n        disk_names=disk_names, output_dir=output_dir, ordered=True)\n\n\ndef _verify_saved_artifacts(table, signatures, batch_rows=ARTIFACT_BATCH_ROWS):\n    with tempfile.TemporaryDirectory(prefix="dl_poc_artifact_verify_") as directory:\n        saved = read_artifacts(table, signatures, disk_names=signatures, output_dir=directory,\n                               batch_rows=batch_rows)\n        if any(artifact_signature(saved[name]) != signature for name, signature in signatures.items()):\n            raise ValueError("Saved artifact read-back differs from the completed run.")\n\n\ndef _artifact_ddl(sql):\n    \n    \n    spark._jvm.net.snowflake.spark.snowflake.Utils.runQuery(dict(sf_options_dl_poc), sql)\n\n\ndef _has_artifact_jvm():\n    """Detect JVM availability before transferring any artifact payload."""\n    try:\n        return getattr(spark, "_jvm", None) is not None\n    except (AttributeError, NotImplementedError):\n        \n        return False\n\n\ndef save_artifacts(table, artifacts, *, chunk_size=50000, batch_rows=ARTIFACT_BATCH_ROWS):\n    """Bounded writes with immutable, verified completed artifacts.\n\n    Classic Spark verifies a pending table before atomic rename. Without a JVM,\n    write directly using errorifexists then append. A failed direct transfer can\n    leave an incomplete final-name table: readers reject its counts/hash and a\n    retry fails closed, requiring a new RUN_ID. Neither mode overwrites a table.\n    """\n    table = _checked_table(table)\n    if not artifacts or not isinstance(batch_rows, int) or batch_rows < 1:\n        raise ValueError("Require artifacts and positive batch_rows")\n    signatures = {name: artifact_signature(blob) for name, blob in artifacts.items()}\n    if table_exists(table):\n        try:\n            _verify_saved_artifacts(table, signatures, batch_rows)\n        except ValueError as exc:\n            raise FileExistsError("Destination contains incomplete or different artifacts; choose a new RUN_ID.") from exc\n        print(f"Existing artifacts verified: DSVC_TAKEDA_TA_PRIVATE.DS_ML.{table}")\n        return\n    atomic_rename = _has_artifact_jvm()\n    staging = table + "__PENDING_" + uuid.uuid4().hex.upper() if atomic_rename else table\n    _checked_table(staging)\n    if not atomic_rename:\n        print("Artifact publication: JVM unavailable; direct immutable batches. "\n              "Interrupted uploads remain invalid and require a new RUN_ID.", flush=True)\n    schema = ("ARTIFACT_NAME STRING, CHUNK_INDEX INT, CHUNK_COUNT INT, "\n              "BYTE_LENGTH LONG, SHA256 STRING, PAYLOAD_BASE64 STRING")\n    rows, first = iter_artifact_rows(artifacts, chunk_size, signatures), True\n    while True:\n        batch = list(islice(rows, batch_rows))\n        if not batch:\n            break\n        frame = spark.createDataFrame(batch, schema=schema)\n        (frame.write.format("snowflake").options(**sf_options_dl_poc)\n         .option("dbtable", staging).option("truncate_columns", "off")\n         .mode("errorifexists" if first else "append").save())\n        first = False\n        del frame, batch\n    _verify_saved_artifacts(staging, signatures, batch_rows)\n    if atomic_rename:\n        \n        \n        _artifact_ddl(f"ALTER TABLE {staging} RENAME TO {table}")\n    print(f"Saved and verified: DSVC_TAKEDA_TA_PRIVATE.DS_ML.{table}")\n', 'pipeline': '"""Audit gates and streamed feature-package persistence for expanded populations."""\nimport hashlib\nimport json\nfrom pathlib import Path\nimport tempfile\n\nimport numpy as np\nimport pandas as pd\n\n\ndef validate_evidence(config):\n    evidence = config.get("cohort_evidence", {})\n    required = ("eligibility_reference", "first_initiation_reference",\n                "negative_followup_reference", "effective_cutoff_reference", "source_snapshot_reference")\n    missing = [key for key in required if not isinstance(evidence.get(key), str) or not evidence[key].strip()]\n    if missing:\n        raise ValueError("Cohort audit binding is incomplete: " + ", ".join(missing))\n    lineage = evidence.get("lineage_status", "reviewed_producer")\n    if lineage == "candidate_as_delivered_development":\n        if (evidence.get("candidate_labels_accepted") is not True or evidence.get("automatic_relabel") is not False\n                or not isinstance(evidence.get("unresolved_rules"), list) or not evidence["unresolved_rules"]):\n            raise ValueError("Candidate development requires explicit acceptance and recorded unresolved rules; no automatic relabeling")\n    elif lineage == "bounded_reconstruction_development":\n        if evidence.get(\'automatic_relabel\') is not True or not evidence.get(\'unresolved_rules\'):\n            raise ValueError(\'Reconstruction requires explicit label authorization and unresolved lineage record\')\n        if not evidence.get(\'reconstruction_sha256\') or evidence.get(\'candidate_labels_accepted\') is not False:\n            raise ValueError(\'Reconstruction must identify its own query, not claim original producer verification\')\n    elif lineage != "reviewed_producer":\n        raise ValueError("Unknown cohort lineage status")\n    sha = evidence.get("producer_sha256")\n    if lineage == "reviewed_producer" or sha is not None:\n        if not isinstance(sha, str) or len(sha) != 64 or any(c not in "0123456789abcdef" for c in sha.lower()):\n            raise ValueError("The reviewed population producer needs its actual SHA-256 fingerprint")\n    if evidence.get("retention_rule") != "all_eligible_preinitiation_snapshots":\n        raise ValueError("Every retained positive snapshot must independently satisfy its forward outcome window")\n    if evidence.get("outcome_days") != 90 or evidence.get("outcome_boundary") != "cutoff < first_event <= cutoff + 90 days":\n        raise ValueError("Forward 90-day first-initiation contract must be explicit")\n    if not config.get("client_universe", {}).get("expected_population"):\n        raise ValueError("Bind independent aggregate and per-split population counts before extraction")\n    if config.get("experiment_phase") not in ("SUMMARY", "TEMPORAL"):\n        raise ValueError("Declare SUMMARY or TEMPORAL before running")\n    if evidence.get("historical_availability") != "retrospective_latest_sources":\n        raise ValueError("This implementation does not prove historical ingestion-time availability")\n    return dict(evidence)\n\n\ndef load_metadata(read_query, read_sf, config, population):\n    validate_evidence(config)\n    cfg = config["client_universe"]\n    name = population.resolve_universe(read_query, cfg["table"], cfg["table_name"])\n    prior = population.frame(read_sf("PATIENT_SPLIT"))\n    incoming = population.frame(read_query("SELECT PATIENT_ID, END_DT, RESP FROM " + name))\n    metadata, audit = population.align_universe(incoming, prior, cfg.get("split_seed", 42), change_policy=cfg.get(\'change_policy\'))\n    population.assert_expected_population(audit, cfg["expected_population"])\n    return metadata, audit\n\n\nNLP_NAMES = {"sequence.npy", "mask.npy", "manifest.json"}\n\n\ndef nlp_identity(metadata, config, implementation, inp, wf):\n    return dict(metadata_hash=inp.frame_hash(metadata), configuration_hash=wf.digest(config),\n                implementation_hash=implementation, fit_scope="train_only", orientation="0=newest")\n\n\ndef save_nlp(result, metadata, config, implementation, prefix, inp, wf, wh):\n    if inp.frame_hash(result["metadata"]) != inp.frame_hash(metadata):\n        raise ValueError("NLP output reordered or changed the population")\n    names = list(result["feature_names"])\n    if not names or len(set(names)) != len(names):\n        raise ValueError("NLP feature order must be nonempty and unique")\n    x, mask = result["X_sequence"], result["mask"]\n    if x.shape != (len(metadata), 12, len(names)) or mask.shape != (len(metadata), 12) or mask.dtype != bool:\n        raise ValueError("NLP output must be N x 12 x F with explicit boolean activity")\n    for start in range(0, len(metadata), 256):\n        a, m = x[start:start+256], mask[start:start+256]\n        if not np.isfinite(a).all() or (a < 0).any() or (a[~m] != 0).any():\n            raise ValueError("NLP values must be finite nonnegative; inactive documents must remain zero")\n    identity = nlp_identity(metadata, config, implementation, inp, wf)\n    fitted, receipt = result["fitted_state"], result["receipt"]\n    expected_patients = inp.train_patient_hash(metadata)\n    train_meta = metadata.loc[metadata.SPLIT.eq("train")]\n    train_requests, _ = inp.make_requests(train_meta, date_policy="month_end_history")\n    all_requests, _ = inp.make_requests(metadata, date_policy="month_end_history")\n    membership = metadata.groupby("PATIENT_ID").SPLIT.first().to_dict()\n    ordered = all_requests.sort_values(["PATIENT_ID", "AS_OF_DATE"]).reset_index(drop=True)\n    request_hash = inp.stable_hash([[i, row.PATIENT_ID, row.AS_OF_DATE.strftime("%Y-%m-%d"), membership[row.PATIENT_ID]]\n                                   for i, row in enumerate(ordered.itertuples(index=False))])\n    for evidence in (fitted, receipt):\n        if (evidence.get("fit_scope") != "train_only" or evidence.get("train_patients_sha256") != expected_patients\n                or evidence.get("train_document_count") != len(train_requests)):\n            raise ValueError("NLP fitted state does not match the frozen TRAIN patient/document scope")\n    if fitted.get("terms") != names or fitted.get("term_order_sha256") != inp.stable_hash(names):\n        raise ValueError("NLP fitted vocabulary differs from its output columns")\n    if fitted.get("extraction_identity", {}).get("requests") != request_hash:\n        raise ValueError("NLP fitted state belongs to different ordered monthly documents or split assignments")\n    manifest = dict(identity, feature_names=names, fitted_state=result["fitted_state"],\n                    receipt=result["receipt"], feature_order_sha256=wf.digest(names),\n                    source_note="Current reference descriptions; retrospective event-date-bounded features",\n                    ranking_kind="TRAIN snapshot-position mean TF-IDF salience; not predictive importance",\n                    train_patients_sha256=expected_patients, train_document_count=len(train_requests))\n    with tempfile.TemporaryDirectory(prefix="monthly_nlp_write_") as directory:\n        files = {"sequence.npy": Path(directory)/"sequence.npy", "mask.npy": Path(directory)/"mask.npy"}\n        np.save(files["sequence.npy"], x, allow_pickle=False)\n        np.save(files["mask.npy"], mask, allow_pickle=False)\n        manifest["payload_hashes"] = {key: wf.file_sha256(path) for key, path in files.items()}\n        wh.save_artifacts(prefix+"_NLP_INPUTS", dict(files, **{"manifest.json": wf.json_bytes(manifest)}))\n    return manifest\n\n\ndef load_nlp(metadata, config, implementation, prefix, inp, wf, wh):\n    owner = wf._InputDirectory()\n    try:\n        payload = wh.read_artifacts(prefix+"_NLP_INPUTS", NLP_NAMES,\n                                    disk_names={"sequence.npy", "mask.npy"}, output_dir=owner.name)\n        manifest = json.loads(payload["manifest.json"])\n        identity = nlp_identity(metadata, config, implementation, inp, wf)\n        if any(manifest.get(k) != v for k, v in identity.items()):\n            raise ValueError("NLP checkpoint population, configuration or implementation mismatch")\n        arrays = {}\n        for key, name in (("X_sequence", "sequence.npy"), ("mask", "mask.npy")):\n            if wf.file_sha256(payload[name]) != manifest["payload_hashes"][name]:\n                raise ValueError("NLP payload content changed")\n            arrays[key] = np.load(payload[name], mmap_mode="r", allow_pickle=False)\n            owner.mappings.append(arrays[key]._mmap)\n        return dict(arrays, metadata=metadata, feature_names=manifest["feature_names"],\n                    manifest=manifest, _owner=owner)\n    except BaseException:\n        owner.cleanup()\n        raise\n\n\ndef release_nlp(package):\n    owner = package.pop("_owner", None)\n    if owner is not None:\n        owner.cleanup()\n\n\ndef nlp_ranking(package):\n    """Streaming TRAIN salience without displaying patient identifiers or histories."""\n    x = package["X_sequence"]\n    rows = np.flatnonzero(package["metadata"].SPLIT.eq("train").to_numpy())\n    sums, nonzero = np.zeros(x.shape[-1]), np.zeros(x.shape[-1], dtype=np.int64)\n    for start in range(0, len(rows), 256):\n        block = x[rows[start:start+256]]\n        sums += block.sum(axis=(0, 1), dtype=np.float64)\n        nonzero += (block > 0).sum(axis=(0, 1))\n    out = pd.DataFrame(dict(Feature=package["feature_names"],\n                           Mean_TFIDF=sums/max(1, len(rows)*12), Nonzero_positions=nonzero))\n    out = out.sort_values(["Mean_TFIDF", "Feature"], ascending=[False, True], kind="stable").reset_index(drop=True)\n    out.insert(0, "Rank", np.arange(1, len(out)+1))\n    return out\n', 'modeling': 'import numpy as np\n\ndef sample_weights(groups, mode="patient"):\n    groups = np.asarray(groups)\n    if groups.ndim != 1 or len(groups) == 0 or any(x is None for x in groups):\n        raise ValueError("Expected nonempty patient IDs")\n    if mode == "snapshot":\n        return np.ones(len(groups), dtype=np.float32)\n    if mode != "patient":\n        raise ValueError("weighting must be patient or snapshot")\n    _, inverse, counts = np.unique(groups.astype(str), return_inverse=True, return_counts=True)\n    w = 1. / counts[inverse]\n    return (w / w.mean()).astype(np.float32)\n\ndef class_weight(y, weights, balance):\n    y, weights = np.asarray(y), np.asarray(weights)\n    if (set(np.unique(y)) != {0, 1} or y.shape != weights.shape\n            or not np.isfinite(weights).all() or (weights <= 0).any()):\n        raise ValueError("Class weighting requires both binary classes and positive aligned weights")\n    ratio = float(weights[y == 0].sum() / weights[y == 1].sum())\n    if balance not in {"none", "sqrt", "full"}:\n        raise ValueError("balance must be none, sqrt or full")\n    return {"none": 1., "sqrt": ratio ** .5, "full": ratio}[balance]\n', 'raw_automl': 'import re, hashlib\n\ndef _feature_name(category, measure):\n    clean = re.sub(r"[^A-Z0-9]+", "_", category.upper()).strip(\'_\')[:160] or "CATEGORY"\n    suffix = hashlib.sha256(category.encode(\'utf-8\')).hexdigest()[:12]\n    return "AML_" + clean + "__" + suffix + ("_COUNT" if measure == "count" else "_RECENT_PCT")\n', 'settings': '\n\nSETTINGS = dict(seeds=[42, 142, 242], n_estimators=1000, learning_rate=.05,\n                num_leaves=31, max_depth=-1, min_child_samples=20,\n                colsample_bytree=.8, subsample=.8, subsample_freq=1,\n                reg_lambda=1., n_jobs=8, stopping_rounds=100,\n                weighting="patient", balance="sqrt", primary_fraction=.1)\n', 'time_lgbm': 'import re\nimport numpy as np\n\nFEATURE_NAMES = [signal + "_" + suffix for signal in ("STUDY", "SLEEP_WAKE")\n                 for suffix in ("MONTHS_SINCE_LAST", "ACTIVE_MONTHS", "RECENT3_MINUS_PRIOR9_MONTHLY_MEAN",\n                                "MONTHLY_SLOPE")] + ["STUDY_M0_5_AND_SLEEP_WAKE_M0_2"]\n\nROLES = {"study": ("PX", {"sleep study", "sleep studies", "level 1 cpt sleep study"}),\n         "diagnosis": ("DX", {"sleep wake disorder", "sleep wake disorders"})}\n\ndef channels(names, exact):\n    if len(set(names)) != len(names) or set(exact) != set(ROLES):\n        raise ValueError("Invalid count dictionary or channel configuration")\n    result = []\n    for role, (family, labels) in ROLES.items():\n        options = [n for n in names if n.startswith(family + "__")]\n        if exact[role]:\n            matches = [n for n in options if n == exact[role]]\n        else:\n            matches = [n for n in options if re.sub("[^a-z0-9]+", " ", n.split("__", 1)[1].lower()).strip() in labels]\n        if len(matches) != 1:\n            candidates = sorted(n for n in options if "sleep" in n.lower())\n            raise ValueError(f"Need one verified {family} count category for {role}. "\n                             f"Set its exact name using the source dictionary. Candidates: {candidates}")\n        result.append(names.index(matches[0]))\n    return result\n\ndef temporal_columns(counts, masks, indices, column_indices):\n    """Read two monthly channels per row block. Position zero ends at snapshot cutoff."""\n    if counts.ndim != 3 or counts.shape[1] != 12 or masks.shape != counts.shape[:2] or masks.dtype != bool:\n        raise ValueError("Expected twelve monthly count positions and a boolean position mask")\n    out = np.full((len(indices), len(FEATURE_NAMES)), np.nan, np.float32)\n    newest_first = np.arange(12, dtype=float)\n    slope_weights = (5.5 - newest_first) / 143.\n    for start in range(0, len(indices), 1024):\n        idx = indices[start:start+1024]\n        complete = masks[idx].all(axis=1)\n        block = out[start:start+len(idx)]\n        recent = []\n        for j, col in enumerate(column_indices):\n            x = counts[idx, :, col]\n            if not np.isfinite(x).all() or (x < 0).any():\n                raise ValueError("Invalid count values")\n            active = x > 0\n            last = np.where(active.any(axis=1), active.argmax(axis=1), 12)\n            block[:, j*4:j*4+4] = np.column_stack((last, active.sum(axis=1),\n                x[:, :3].mean(axis=1)-x[:, 3:].mean(axis=1), x @ slope_weights))\n            recent.append(active[:, :6 if j == 0 else 3].any(axis=1))\n        block[:, -1] = recent[0] & recent[1]\n        block[~complete] = np.nan\n    return out\n', 'cache': '"""Read-only, bounded handoff of completed raw-feature checkpoints."""\nfrom __future__ import annotations\n\nimport hashlib\nimport json\nimport math\nimport os\nfrom pathlib import Path\nimport sqlite3\nfrom contextlib import closing\nfrom functools import lru_cache\n\nimport pandas as pd\n\ntry:\n    from experiments.automl_sequence import inputs\nexcept ModuleNotFoundError:\n    inputs = None\n\n\nCOMPACT_COLUMNS = ("p", "d", "k", "c", "v", "r", "e", "a", "ac", "chunk_no")\nSOURCE_COLUMNS = ("p", "d", "k", "c", "raw_json", "v", "r", "e", "a", "ac")\nSCHEMA = 1\n\n\ndef _json_bytes(value):\n    return json.dumps(value, sort_keys=True, separators=(",", ":"), allow_nan=False).encode()\n\n\ndef _hash_file(path):\n    digest = hashlib.sha256()\n    with open(path, "rb") as handle:\n        for part in iter(lambda: handle.read(1024 * 1024), b""):\n            digest.update(part)\n    return digest.hexdigest()\n\n\ndef _atomic(path, payload):\n    path = Path(path)\n    temporary = path.with_name(path.name + ".partial")\n    with temporary.open("wb") as handle:\n        handle.write(payload)\n        handle.flush()\n        os.fsync(handle.fileno())\n    os.replace(temporary, path)\n\n\ndef _readonly(path):\n    path = Path(path).resolve(strict=True)\n    connection = sqlite3.connect(path.as_uri() + "?mode=ro", uri=True, timeout=2,\n                                 isolation_level=None)\n    connection.execute("PRAGMA query_only=ON")\n    connection.execute("PRAGMA busy_timeout=2000")\n    return connection\n\n\ndef _read_source_state(connection):\n    state = dict(connection.execute("SELECT key,value FROM state WHERE key IN (\'identity\',\'columns\')"))\n    if set(state) != {"identity", "columns"}:\n        raise ValueError("No completed extraction checkpoint identity and column declaration")\n    identity = json.loads(state["identity"])\n    columns = json.loads(state["columns"])\n    required = {"PATIENT_ID", "AS_OF_DATE", "ROW_KIND", "CATEGORY", "VALUE", "RECENT_VALUE",\n                "MAX_EVENT_DATE", "AGE", "AGE_CONFLICT"}\n    if identity.get("schema") != 1 or not required.issubset(columns) or len(columns) != len(set(columns)):\n        raise ValueError("Unsupported raw checkpoint schema")\n    receipts = [dict(chunk_no=int(n), row_count=int(count), sha256=sha)\n                for n, count, sha in connection.execute("SELECT chunk_no,row_count,sha256 FROM chunks ORDER BY chunk_no")]\n    if not receipts:\n        raise ValueError("No fully committed chunks are available")\n    if [x["chunk_no"] for x in receipts] != list(range(len(receipts))):\n        raise ValueError("Completed chunk receipts must form a contiguous prefix")\n    if any(x["row_count"] <= 0 or len(x["sha256"]) != 64 for x in receipts):\n        raise ValueError("Invalid completed-chunk receipt")\n    return state, identity, receipts\n\n\ndef _pages(connection, chunk_no, columns, page_rows):\n    """Finish each short SELECT before processing its page; no long source transaction."""\n    last = None\n    names = ",".join(columns)\n    while True:\n        sql = "SELECT " + names + " FROM raw WHERE chunk_no=?"\n        parameters = [chunk_no]\n        if last is not None:\n            sql += " AND (p,d,k,c) > (?,?,?,?)"\n            parameters.extend(last)\n        sql += " ORDER BY p,d,k,c LIMIT ?"\n        parameters.append(page_rows)\n        cursor = connection.execute(sql, parameters)\n        try:\n            rows = cursor.fetchall()\n        finally:\n            cursor.close()\n        if not rows:\n            return\n        last = tuple(rows[-1][:4])\n        yield rows\n\n\ndef _compact_digest(connection, chunk_no, page_rows=4096):\n    digest = hashlib.sha256(_json_bytes(COMPACT_COLUMNS))\n    count = 0\n    for page in _pages(connection, chunk_no, COMPACT_COLUMNS, page_rows):\n        for row in page:\n            digest.update(_json_bytes(row) + b"\\n")\n        count += len(page)\n    return count, digest.hexdigest()\n\n\n@lru_cache(maxsize=65536)\ndef _validate_dates(date, event):\n    cutoff = pd.Timestamp(date)\n    if cutoff.strftime("%Y-%m-%d") != date:\n        raise ValueError("Raw checkpoint dates must be exact days")\n    if event is not None:\n        event_date = pd.Timestamp(event)\n        if event_date.strftime("%Y-%m-%d") != event or not cutoff - pd.DateOffset(months=12) <= event_date <= cutoff:\n            raise ValueError("Cached source event exceeds its feature window")\n\n\ndef _validate_row(row):\n    patient, date, kind, category, value, recent, event, age, conflict, chunk_no = row\n    if not isinstance(patient, str) or not patient or kind not in {"COVERAGE", "FEATURE"}:\n        raise ValueError("Invalid cached raw record key/kind")\n    _validate_dates(date, event)\n    if kind == "FEATURE":\n        if not category or event is None or value is None or recent is None:\n            raise ValueError("Feature rows require category, counts and event evidence")\n        if not all(math.isfinite(x) and x == math.floor(x) for x in (value, recent)) or not 0 <= recent <= value or value <= 0:\n            raise ValueError("Invalid annual/recent distinct counts")\n    elif category != "" or value is not None or recent is not None:\n        raise ValueError("Invalid coverage row")\n    if age is not None and (not math.isfinite(age) or age < 0):\n        raise ValueError("Invalid cached age")\n    if conflict is not None and not math.isfinite(conflict):\n        raise ValueError("Invalid cached age-conflict indicator")\n\n\ndef _coverage_check(connection, chunk_no, requests):\n    returned = list(connection.execute("SELECT p,d FROM raw WHERE chunk_no=? AND k=\'COVERAGE\' ORDER BY p,d", (chunk_no,)))\n    expected = [(p, d.strftime("%Y-%m-%d")) for p, d in requests.itertuples(index=False, name=None)]\n    if returned != expected:\n        raise ValueError("Completed chunk does not cover exactly its original requested keys")\n    invalid = connection.execute("SELECT 1 FROM raw f LEFT JOIN raw c ON f.p=c.p AND f.d=c.d AND c.k=\'COVERAGE\' "\n                                 "AND f.chunk_no=c.chunk_no WHERE f.chunk_no=? AND f.k=\'FEATURE\' "\n                                 "AND (c.p IS NULL OR c.e IS NULL) LIMIT 1", (chunk_no,)).fetchone()\n    if invalid:\n        raise ValueError("Feature rows lack observed coverage evidence")\n\n\ndef _base_path(location):\n    path = Path(location).resolve()\n    return path.parent if path.name == "manifest.json" else path\n\n\ndef freeze_cache(source_path, destination, metadata, *, chunk_limit=None, page_rows=4096,\n                 previous_destination=None):\n    """Freeze committed SUMMARY chunks without changing the full extraction.\n\n    Run after the writer has completed a chunk, then let it resume. A frozen\n    export plan ignores later commits. Repeating this call resumes that same\n    plan; use a new destination for additional chunks. Existing exported chunks\n    can be imported with previous_destination without rereading raw source rows.\n    """\n    if inputs is None:\n        raise RuntimeError("Inject the existing inputs module")\n    if type(page_rows) is not int or not 1 <= page_rows <= 65536:\n        raise ValueError("page_rows must be between 1 and 65536")\n    if chunk_limit is not None and (type(chunk_limit) is not int or chunk_limit < 1):\n        raise ValueError("chunk_limit must be a positive integer")\n    source = Path(source_path).resolve(strict=True)\n    dest = _base_path(destination)\n    if dest.is_relative_to(source.parent) or source.is_relative_to(dest):\n        raise ValueError("Use an isolated export directory outside the extraction checkpoint directory")\n    meta = inputs.normalize_snapshots(metadata)\n    requests, _ = inputs.make_requests(meta, steps=1, date_policy="month_end_history")\n    metadata_hash = inputs.frame_hash(meta)\n    source_db = _readonly(source)\n    target = None\n    previous_db = None\n    try:\n        state, identity, receipts = _read_source_state(source_db)\n        chunk_size = identity.get("chunk_size")\n        if type(chunk_size) is not int or not 1 <= chunk_size <= 10000:\n            raise ValueError("Invalid source request chunk size")\n        if identity.get("requests_sha256") != inputs.frame_hash(requests):\n            raise ValueError("Source request identity differs: only the matching SUMMARY population is supported")\n        if identity.get("train_patients_sha256") != inputs.train_patient_hash(meta):\n            raise ValueError("Source TRAIN patient membership differs")\n        number_chunks = (len(requests) + chunk_size - 1) // chunk_size\n        if len(receipts) > number_chunks:\n            raise ValueError("Source contains unexpected chunk indices")\n        dest.mkdir(parents=True, exist_ok=True)\n        plan_path = dest / "frozen_plan.json"\n        if plan_path.exists():\n            plan = json.loads(plan_path.read_text(encoding="utf-8"))\n            if (plan["identity"] != identity or plan["source_columns_json"] != state["columns"]\n                    or plan["parent_metadata_sha256"] != metadata_hash):\n                raise ValueError("Frozen handoff identity changed; choose a new export directory")\n            if chunk_limit is not None and chunk_limit != len(plan["chunks"]):\n                raise ValueError("Cannot change the frozen chunk boundary in an existing export")\n            if receipts[:len(plan["chunks"])] != plan["chunks"]:\n                raise ValueError("Previously frozen source receipts changed or disappeared")\n        else:\n            if chunk_limit is not None and chunk_limit > len(receipts):\n                raise ValueError("Requested chunk boundary is not fully committed")\n            chosen = receipts[:chunk_limit] if chunk_limit else receipts\n            meta_bytes = meta.to_csv(index=False, lineterminator="\\n", date_format="%Y-%m-%d").encode()\n            request_file = requests.copy()\n            request_file["chunk_no"] = request_file.index // chunk_size\n            request_bytes = request_file.to_csv(index=False, lineterminator="\\n", date_format="%Y-%m-%d").encode()\n            plan = dict(schema=SCHEMA, identity=identity, source_columns_json=state["columns"],\n                        parent_metadata_sha256=metadata_hash, chunks=chosen, request_steps=1,\n                        total_requests=len(requests), frozen_request_count=min(len(chosen)*chunk_size, len(requests)),\n                        analysis_db="analysis.sqlite", metadata_file="metadata.csv", requests_file="requests.csv",\n                        metadata_file_sha256=hashlib.sha256(meta_bytes).hexdigest(),\n                        requests_file_sha256=hashlib.sha256(request_bytes).hexdigest(),\n                        sampling="completed sorted request prefix; complete patients only; not a random sample",\n                        source_vintage="checkpoint cache may combine source vintages; historical availability unresolved")\n            _atomic(dest / "metadata.csv", meta_bytes)\n            _atomic(dest / "requests.csv", request_bytes)\n            _atomic(plan_path, _json_bytes(plan))\n        for filename, key in (("metadata.csv", "metadata_file_sha256"), ("requests.csv", "requests_file_sha256")):\n            if _hash_file(dest / filename) != plan[key]:\n                raise ValueError("Frozen metadata or request file checksum mismatch")\n        previous = None\n        if previous_destination is not None:\n            previous = load_frozen_cache(previous_destination)\n            if (previous["identity"] != identity or previous["parent_metadata_sha256"] != metadata_hash\n                    or previous["source_columns_json"] != state["columns"]):\n                raise ValueError("Previous export has a different parent identity")\n            previous_db = _readonly(previous["analysis_db_path"])\n        target = sqlite3.connect(dest / "analysis.sqlite", timeout=2)\n        target.execute("PRAGMA temp_store=FILE")\n        target.execute("PRAGMA cache_size=-16384")\n        with target:\n            target.execute("CREATE TABLE IF NOT EXISTS raw(p TEXT,d TEXT,k TEXT,c TEXT,v REAL,r REAL,e TEXT,a REAL,ac REAL,chunk_no INTEGER,PRIMARY KEY(p,d,k,c)) WITHOUT ROWID")\n            target.execute("CREATE INDEX IF NOT EXISTS raw_by_chunk ON raw(chunk_no,p,d,k,c)")\n            target.execute("CREATE TABLE IF NOT EXISTS chunks(chunk_no INTEGER PRIMARY KEY,row_count INTEGER,source_sha256 TEXT,export_sha256 TEXT)")\n        saved_chunks = dict((n, (count, source_sha, export_sha)) for n, count, source_sha, export_sha in target.execute("SELECT * FROM chunks"))\n        allowed = {x["chunk_no"] for x in plan["chunks"]}\n        if set(saved_chunks) - allowed or target.execute("SELECT 1 FROM raw LEFT JOIN chunks USING(chunk_no) WHERE chunks.chunk_no IS NULL LIMIT 1").fetchone():\n            raise ValueError("Analysis cache contains unreceipted or unexpected rows")\n        completed = []\n        previous_receipts = {} if previous is None else {x["chunk_no"]: x for x in previous["chunks"]}\n        for receipt in plan["chunks"]:\n            chunk_no = receipt["chunk_no"]\n            batch = requests.iloc[chunk_no*chunk_size:(chunk_no+1)*chunk_size]\n            saved = saved_chunks.get(chunk_no)\n            if saved:\n                count, exported = _compact_digest(target, chunk_no, page_rows)\n                if saved != (count, receipt["sha256"], exported) or count != receipt["row_count"]:\n                    raise ValueError("Analysis cache checkpoint checksum mismatch")\n                _coverage_check(target, chunk_no, batch)\n                mode = "reused"\n            else:\n                original_digest = hashlib.sha256(state["columns"].encode())\n                export_digest = hashlib.sha256(_json_bytes(COMPACT_COLUMNS))\n                count = 0\n                inherited = previous_receipts.get(chunk_no)\n                if inherited and any(inherited[k] != receipt[k] for k in ("chunk_no", "row_count", "sha256")):\n                    raise ValueError("Previous export chunk differs from source receipt")\n                try:\n                    target.execute("BEGIN IMMEDIATE")\n                    reader = previous_db if inherited else source_db\n                    columns = COMPACT_COLUMNS if inherited else SOURCE_COLUMNS\n                    for page in _pages(reader, chunk_no, columns, page_rows):\n                        compact = []\n                        for row in page:\n                            if inherited:\n                                item = row\n                            else:\n                                original = list(row)\n                                original[4] = original[4].hex()\n                                original_digest.update(json.dumps(original, separators=(",", ":"), allow_nan=False).encode() + b"\\n")\n                                item = tuple(row[:4]) + tuple(row[5:]) + (chunk_no,)\n                            _validate_row(item)\n                            export_digest.update(_json_bytes(item) + b"\\n")\n                            compact.append(item)\n                        target.executemany("INSERT INTO raw VALUES (?,?,?,?,?,?,?,?,?,?)", compact)\n                        count += len(compact)\n                    exported = export_digest.hexdigest()\n                    if count != receipt["row_count"] or (not inherited and original_digest.hexdigest() != receipt["sha256"]):\n                        raise ValueError("Source checkpoint content checksum mismatch")\n                    if inherited and exported != inherited["export_sha256"]:\n                        raise ValueError("Previous export content checksum mismatch")\n                    _coverage_check(target, chunk_no, batch)\n                    target.execute("INSERT INTO chunks VALUES (?,?,?,?)", (chunk_no, count, receipt["sha256"], exported))\n                    target.commit()\n                except BaseException:\n                    target.rollback()\n                    raise\n                mode = "imported" if inherited else "exported"\n            completed.append(dict(receipt, export_sha256=exported))\n            print("Analysis chunk", chunk_no+1, "/", len(plan["chunks"]), mode, "rows", count, flush=True)\n        end_state, end_identity, end_receipts = _read_source_state(source_db)\n        if end_identity != identity or end_state != state or end_receipts[:len(plan["chunks"])] != plan["chunks"]:\n            raise ValueError("Source checkpoint changed while exporting frozen chunks")\n        with target:\n            target.execute("CREATE INDEX IF NOT EXISTS raw_by_category ON raw(k,c,p,d)")\n        manifest = dict(plan, chunks=completed, status="verified", plan_sha256=_hash_file(plan_path),\n                        compact_columns=list(COMPACT_COLUMNS))\n        _atomic(dest / "manifest.json", _json_bytes(manifest))\n        return dest / "manifest.json"\n    finally:\n        if previous_db is not None:\n            previous_db.close()\n        if target is not None:\n            target.close()\n        source_db.close()\n\n\ndef load_frozen_cache(location, *, verify_chunks=True):\n    """Verify a completed handoff without opening or depending on its source cache."""\n    dest = _base_path(location)\n    manifest = json.loads((dest / "manifest.json").read_text(encoding="utf-8"))\n    if manifest.get("schema") != SCHEMA or manifest.get("status") != "verified" or manifest.get("request_steps") != 1:\n        raise ValueError("Incomplete or unsupported analysis handoff")\n    for filename, key in (("metadata.csv", "metadata_file_sha256"), ("requests.csv", "requests_file_sha256"), ("frozen_plan.json", "plan_sha256")):\n        if _hash_file(dest / filename) != manifest[key]:\n            raise ValueError("Analysis handoff file checksum mismatch")\n    plan = json.loads((dest / "frozen_plan.json").read_text(encoding="utf-8"))\n    if any(manifest.get(key) != value for key, value in plan.items() if key != "chunks"):\n        raise ValueError("Analysis manifest differs from frozen export plan")\n    if [{k: row[k] for k in ("chunk_no", "row_count", "sha256")} for row in manifest["chunks"]] != plan["chunks"]:\n        raise ValueError("Analysis manifest changed its frozen chunk list")\n    if manifest.get("compact_columns") != list(COMPACT_COLUMNS):\n        raise ValueError("Unexpected analysis cache columns")\n    with closing(_readonly(dest / "analysis.sqlite")) as connection:\n        actual = list(connection.execute("SELECT chunk_no,row_count,source_sha256,export_sha256 FROM chunks ORDER BY chunk_no"))\n        expected = [(x["chunk_no"], x["row_count"], x["sha256"], x["export_sha256"]) for x in manifest["chunks"]]\n        if actual != expected or connection.execute("SELECT 1 FROM raw LEFT JOIN chunks USING(chunk_no) WHERE chunks.chunk_no IS NULL LIMIT 1").fetchone():\n            raise ValueError("Analysis cache receipt mismatch")\n        if verify_chunks:\n            for receipt in manifest["chunks"]:\n                if _compact_digest(connection, receipt["chunk_no"]) != (receipt["row_count"], receipt["export_sha256"]):\n                    raise ValueError("Analysis cache content checksum mismatch")\n    return dict(manifest, analysis_db_path=str(dest / "analysis.sqlite"), metadata_path=str(dest / "metadata.csv"),\n                requests_path=str(dest / "requests.csv"), manifest_path=str(dest / "manifest.json"))\n\n\ndef complete_patient_metadata(location):\n    """Keep every parent snapshot for a patient, or none; never infer row order."""\n    manifest = location if isinstance(location, dict) else load_frozen_cache(location)\n    meta = inputs.normalize_snapshots(pd.read_csv(manifest["metadata_path"], dtype={"PATIENT_ID": str}))\n    if inputs.frame_hash(meta) != manifest["parent_metadata_sha256"]:\n        raise ValueError("Parent metadata identity mismatch")\n    with closing(_readonly(manifest["analysis_db_path"])) as connection:\n        coverage = pd.read_sql_query("SELECT p AS PATIENT_ID,d AS END_DT FROM raw WHERE k=\'COVERAGE\' ORDER BY p,d", connection)\n    coverage["END_DT"] = pd.to_datetime(coverage.END_DT)\n    joined = meta.merge(coverage.assign(available=True), on=["PATIENT_ID", "END_DT"], how="left", validate="one_to_one")\n    complete = joined.groupby("PATIENT_ID", sort=False).available.count().eq(joined.groupby("PATIENT_ID", sort=False).size())\n    return meta.loc[meta.PATIENT_ID.isin(complete.index[complete])].reset_index(drop=True)\n', 'analysis': '"""Frozen-cache development comparisons with bounded features and resumable fits."""\nfrom __future__ import annotations\n\nimport gc\nimport hashlib\nimport json\nimport os\nimport sqlite3\nimport time\nfrom contextlib import closing\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\n\n\nSCHEMA = \'cached_pilot_report_v1\'\nSOURCE_SHA256 = \'cached_pilot_analysis_v1\'\nSETTINGS = dict(FULL_SETTINGS, seeds=[42], n_estimators=300, n_jobs=2)\nSETTINGS.pop(\'stopping_rounds\', None)\nKEYS = [\'PATIENT_ID\', \'END_DT\']\nARMS = {\'AUTOML\': [\'automl\'], \'AUTOML_TIME\': [\'automl\', \'time\'],\n        \'AUTOML_NLP\': [\'automl\', \'nlp\'], \'AUTOML_TIME_NLP\': [\'automl\', \'time\', \'nlp\']}\n\n\ndef _atomic_json(path, value):\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    tmp = path.with_name(path.name + \'.partial\')\n    tmp.write_bytes(workflow.json_bytes(value))\n    os.replace(tmp, path)\n\n\ndef _atomic_bytes(path, value):\n    path = Path(path)\n    tmp = path.with_name(path.name + \'.partial\')\n    tmp.write_bytes(value)\n    os.replace(tmp, path)\n\n\ndef _file_hash(path):\n    result = hashlib.sha256()\n    with open(path, \'rb\') as stream:\n        for block in iter(lambda: stream.read(1024 * 1024), b\'\'):\n            result.update(block)\n    return result.hexdigest()\n\n\ndef _records(meta):\n    out = meta.copy()\n    out[\'END_DT\'] = out.END_DT.dt.strftime(\'%Y-%m-%d\')\n    return out.to_dict(\'records\')\n\n\ndef _key_hash(meta):\n    return workflow.digest([[str(p), pd.Timestamp(d).strftime(\'%Y-%m-%d\')]\n                            for p, d in meta[KEYS].itertuples(index=False, name=None)])\n\n\ndef population_summary(metadata):\n    rows = []\n    for split, frame in [(\'all\', metadata)] + list(metadata.groupby(\'SPLIT\', sort=True)):\n        rows.append(dict(Split=split.upper(), Snapshots=len(frame), Patients=frame.PATIENT_ID.nunique(),\n                         Positive_snapshots=int(frame.RESP.sum()),\n                         Positive_patients=frame.loc[frame.RESP.eq(1), \'PATIENT_ID\'].nunique(),\n                         First_date=str(frame.END_DT.min().date()), Last_date=str(frame.END_DT.max().date())))\n    return pd.DataFrame(rows)\n\n\ndef _load_preparation(path):\n    path = Path(path)\n    if path.is_dir():\n        path = path / \'prepared.json\'\n    result = json.loads(path.read_text(encoding=\'utf-8\'))\n    if result.get(\'schema\') != \'cached_pilot_preparation_v1\':\n        raise ValueError(\'Unsupported prior preparation\')\n    unsigned = {k:v for k,v in result.items() if k != \'sha256\'}\n    if result.get(\'sha256\') != workflow.digest(unsigned):\n        raise ValueError(\'Preparation receipt hash mismatch\')\n    return result\n\n\ndef _assert_classes(meta):\n    for split in (\'train\', \'validation\'):\n        frame = meta.loc[meta.SPLIT.eq(split)]\n        if len(frame) == 0 or set(frame.RESP.unique()) != {0, 1}:\n            raise ValueError(f\'{split}: complete-patient pilot needs both classes; wait for more committed chunks\')\n    if meta.SPLIT.eq(\'test\').any():\n        raise ValueError(\'TEST must remain unused\')\n\n\ndef _read_complete_metadata(destination):\n    complete = cache.complete_patient_metadata(destination)\n    return inputs.normalize_snapshots(complete)\n\n\ndef prepare(cache_manifest, output_root, *, previous_version=None, max_features=512):\n    """Freeze complete TRAIN patients and an initial fixed development panel."""\n    if type(max_features) is not int or max_features < 3:\n        raise ValueError(\'Expected a pair-preserving vocabulary cap of at least three\')\n    manifest = cache.load_frozen_cache(cache_manifest)\n    directory = Path(cache_manifest)\n    if directory.is_file():\n        directory = directory.parent\n    db_path = Path(manifest.get(\'analysis_db_path\', directory / manifest.get(\'analysis_db\', \'analysis.sqlite\')))\n    meta = _read_complete_metadata(manifest)\n    meta = meta.loc[meta.SPLIT.isin([\'train\', \'validation\'])].reset_index(drop=True)\n    previous = _load_preparation(previous_version) if previous_version is not None else None\n    if previous is not None:\n        if previous.get(\'settings\') != SETTINGS or previous.get(\'implementation\') != SOURCE_SHA256:\n            raise ValueError(\'Refresh changed frozen pilot settings or implementation; start a separate experiment\')\n        if previous[\'source_identity\'] != manifest[\'identity\'] or previous[\'parent_metadata_sha256\'] != manifest[\'parent_metadata_sha256\']:\n            raise ValueError(\'Refresh must use the same parent source, metadata, labels and split\')\n        if previous[\'max_features\'] != max_features:\n            raise ValueError(\'Refresh changed the vocabulary cap\')\n        old_val = inputs.normalize_snapshots(pd.DataFrame(previous[\'validation_records\']))\n        available = pd.MultiIndex.from_frame(meta[KEYS])\n        index = available.get_indexer(pd.MultiIndex.from_frame(old_val[KEYS]))\n        if (index < 0).any() or _records(meta.iloc[index].reset_index(drop=True)) != _records(old_val):\n            raise ValueError(\'Frozen validation population or labels changed\')\n        old_train = inputs.normalize_snapshots(pd.DataFrame(previous[\'training_records\']))\n        ti = available.get_indexer(pd.MultiIndex.from_frame(old_train[KEYS]))\n        if (ti < 0).any() or _records(meta.iloc[ti].reset_index(drop=True)) != _records(old_train):\n            raise ValueError(\'Refresh lost or changed prior TRAIN rows\')\n        meta = pd.concat([meta.loc[meta.SPLIT.eq(\'train\')].sort_values(KEYS,kind=\'stable\'), old_val], ignore_index=True)\n    else:\n        \n        meta = meta.sort_values(KEYS, kind=\'stable\').reset_index(drop=True)\n    _assert_classes(meta)\n    train = meta.SPLIT.eq(\'train\').to_numpy()\n    val = meta.SPLIT.eq(\'validation\').to_numpy()\n    output = Path(output_root).resolve()\n    output.mkdir(parents=True, exist_ok=True)\n    source_identity = dict(manifest[\'identity\'])\n    with closing(sqlite3.connect(db_path.as_uri() + \'?mode=ro\', uri=True)) as db:\n        db.execute(\'PRAGMA query_only=ON\')\n        train_patients = set(meta.loc[train, \'PATIENT_ID\'])\n        support = {}\n        for category, patient in db.execute("SELECT DISTINCT c,p FROM raw WHERE k=\'FEATURE\' ORDER BY c,p"):\n            if patient in train_patients:\n                support[category] = support.get(category, 0) + 1\n        if previous:\n            categories = list(previous[\'categories\'])\n        else:\n            categories = sorted(support, key=lambda name: (-support[name], name))[:(max_features-1)//2]\n        specs = sorted([dict(name=raw_automl._feature_name(category, measure), category=category, measure=measure)\n                        for category in categories for measure in (\'count\', \'recent_pct\')]\n                       + [dict(name=\'AML_AGE_YEARS\', category=\'AGE\', measure=\'age\')], key=lambda s:s[\'name\'])\n        names = [item[\'name\'] for item in specs] + [\'AUTOML_CURRENT_AVAILABLE\']\n        if len(set(names)) != len(names):\n            raise ValueError(\'Feature name collision\')\n        if previous and names != previous[\'feature_names\']:\n            raise ValueError(\'Refresh changed frozen feature definitions\')\n        lookup = {(p, d.strftime(\'%Y-%m-%d\')):i for i,(p,d) in enumerate(meta[KEYS].itertuples(index=False,name=None))}\n        x_path = output / \'automl.npy\'\n        tmp_path = output / \'automl.partial.npy\'\n        x = np.lib.format.open_memmap(tmp_path, mode=\'w+\', dtype=np.float32, shape=(len(meta), len(names)))\n        x[:] = np.nan\n        x[:, -1] = 0\n        seen = set()\n        age_col = names.index(\'AML_AGE_YEARS\')\n        for p,d,e,a,ac in db.execute("SELECT p,d,e,a,ac FROM raw WHERE k=\'COVERAGE\' ORDER BY p,d"):\n            row = lookup.get((p,d))\n            if row is None:\n                continue\n            if row in seen:\n                raise ValueError(\'Duplicate coverage key\')\n            seen.add(row)\n            if e is not None:\n                event = pd.Timestamp(e)\n                cutoff = pd.Timestamp(d)\n                if not cutoff-pd.DateOffset(months=12) <= event <= cutoff:\n                    raise ValueError(\'Coverage event outside feature window\')\n                x[row, :-1] = 0\n                x[row, -1] = 1\n                if a is not None and (not np.isfinite(a) or a < 0):\n                    raise ValueError(\'Invalid raw age\')\n                x[row, age_col] = np.nan if a is None or (ac is not None and ac != 0) else a\n        if len(seen) != len(meta):\n            raise ValueError(\'Incomplete requested coverage in frozen cache\')\n        by_category = {}\n        for j,spec in enumerate(specs):\n            if spec[\'measure\'] != \'age\':\n                by_category.setdefault(spec[\'category\'], {})[spec[\'measure\']] = j\n        for category, columns in by_category.items():\n            cursor = db.execute("SELECT p,d,v,r,e FROM raw WHERE k=\'FEATURE\' AND c=? ORDER BY p,d", (category,))\n            while True:\n                batch = cursor.fetchmany(4096)\n                if not batch:\n                    break\n                matched=[(lookup[(p,d)],d,v,r,e) for p,d,v,r,e in batch if (p,d) in lookup]\n                if not matched: continue\n                part=pd.DataFrame(matched,columns=[\'i\',\'d\',\'v\',\'r\',\'e\'])\n                rows=part.i.to_numpy(dtype=np.int64)\n                annual=part.v.to_numpy(dtype=float);recent=part.r.to_numpy(dtype=float)\n                if (not np.isfinite(annual).all() or not np.isfinite(recent).all()\n                    or (annual <= 0).any() or (recent < 0).any() or (recent > annual).any()\n                    or (annual != np.floor(annual)).any() or (recent != np.floor(recent)).any()\n                    or (x[rows,-1] != 1).any()):\n                    raise ValueError(\'Invalid raw counts or unobserved feature row\')\n                cutoff=pd.to_datetime(part.d);event=pd.to_datetime(part.e)\n                if event.isna().any() or (event > cutoff).any() or (event < cutoff-pd.DateOffset(months=12)).any():\n                    raise ValueError(\'Feature event outside cutoff\')\n                x[rows,columns[\'count\']]=annual\n                x[rows,columns[\'recent_pct\']]=np.where((annual>6)&(recent>2),np.floor(100.*recent/annual+.5),0.)\n        x.flush()\n        x._mmap.close()\n        del x\n        gc.collect()\n    x = np.load(tmp_path, mmap_mode=\'r\', allow_pickle=False)\n    if any(np.isinf(x[start:start+4096]).any() for start in range(0,len(x),4096)):\n        raise ValueError(\'Float32 conversion overflow\')\n    val_hash = workflow.array_sha256(x[val])\n    if previous and val_hash != previous[\'validation_automl_sha256\']:\n        raise ValueError(\'Frozen validation feature values changed\')\n    receipt = dict(schema=\'cached_pilot_preparation_v1\', source_identity=source_identity,\n                   parent_metadata_sha256=manifest[\'parent_metadata_sha256\'],\n                   frozen_chunks=manifest[\'chunks\'], cache_manifest_sha256=_file_hash(directory/\'manifest.json\'),\n                   max_features=max_features, categories=categories, feature_names=names, feature_specs=specs,\n                   category_support_scope=\'complete pilot TRAIN patients only; labels unused\',\n                   support_at_creation={c:int(support.get(c,0)) for c in categories},\n                   training_records=_records(meta.loc[train]), validation_records=_records(meta.loc[val]),\n                   training_keys_sha256=_key_hash(meta.loc[train]), evaluation_keys_sha256=_key_hash(meta.loc[val]),\n                   evaluation_labels_sha256=workflow.digest(meta.loc[val,\'RESP\'].tolist()),\n                   validation_automl_sha256=val_hash, automl_sha256=workflow.array_sha256(x),\n                   previous_version=None if previous is None else previous[\'sha256\'],\n                   sampling=\'completed patient prefix of sorted extraction requests; not a random sample\',\n                   missingness=\'unobserved source window stays NaN; observed absent category is zero\',\n                   settings=SETTINGS, implementation=SOURCE_SHA256)\n    receipt[\'sha256\'] = workflow.digest(receipt)\n    existing = output/\'prepared.json\'\n    if existing.exists() and _load_preparation(existing) != receipt:\n        x._mmap.close()\n        raise ValueError(\'Output version already contains a different preparation; choose a new version folder\')\n    x._mmap.close()\n    if existing.exists():\n        saved=np.load(x_path,mmap_mode=\'r\',allow_pickle=False)\n        try:\n            if workflow.array_sha256(saved) != receipt[\'automl_sha256\']:\n                raise ValueError(\'Saved AutoML matrix hash mismatch\')\n        finally:saved._mmap.close()\n        tmp_path.unlink()\n    else:\n        os.replace(tmp_path, x_path)\n    x = np.load(x_path, mmap_mode=\'r\', allow_pickle=False)\n    _atomic_json(existing, receipt)\n    _atomic_bytes(output/\'metadata.csv\', meta.to_csv(index=False, lineterminator=\'\\n\').encode())\n    return dict(metadata=meta, X_automl=x, feature_names=names, train_indices=np.flatnonzero(train),\n                validation_indices=np.flatnonzero(val), preparation=receipt, path=str(output),\n                previous_version=None if previous_version is None else str(previous_version))\n\n\ndef _align_group(prepared, group, name):\n    if not isinstance(group, dict) or group.get(\'provenance\',{}).get(\'verified\') is not True:\n        raise ValueError(f\'{name}: verified provenance is required\')\n    provenance = group[\'provenance\']\n    if name == \'nlp\' and (provenance.get(\'fit_scope\') != \'train_only\'\n                         or provenance.get(\'fitted_on\') not in {\'full_parent_train\',\'pilot_train\'}):\n        raise ValueError(\'NLP encoder fitting scope must be verified TRAIN only\')\n    source = inputs.normalize_snapshots(group[\'metadata\'])\n    target = prepared[\'metadata\']\n    indices = pd.MultiIndex.from_frame(source[KEYS]).get_indexer(pd.MultiIndex.from_frame(target[KEYS]))\n    if (indices < 0).any() or _records(source.iloc[indices].reset_index(drop=True)) != _records(target):\n        raise ValueError(f\'{name}: missing keys, labels or split mismatch\')\n    values = group[\'values\']\n    names = list(group[\'names\'])\n    if values.ndim != 2 or values.shape != (len(source),len(names)) or len(set(names)) != len(names) or not names:\n        raise ValueError(f\'{name}: invalid feature matrix or names\')\n    output = Path(prepared[\'path\'])/(name+\'.aligned.npy\')\n    x = np.lib.format.open_memmap(output, mode=\'w+\', dtype=np.float32, shape=(len(target),len(names)))\n    for start in range(0,len(target),2048):\n        block = np.asarray(values[indices[start:start+2048]], dtype=np.float32)\n        if np.isinf(block).any():\n            raise ValueError(f\'{name}: infinite feature value\')\n        x[start:start+len(block)] = block\n    x.flush()\n    return dict(values=x, names=names, provenance=provenance,\n                evaluation_sha256=workflow.array_sha256(x[prepared[\'validation_indices\']]),\n                full_sha256=workflow.array_sha256(x))\n\n\ndef memory_guard(required_bytes=0, min_free_gib=2):\n    import psutil\n    available = int(psutil.virtual_memory().available)\n    needed = int(required_bytes) + int(min_free_gib*1024**3)\n    if available < needed:\n        raise MemoryError(f\'Pause analysis to preserve extraction: available {available/1024**3:.2f} GiB; \'\n                          f\'need {needed/1024**3:.2f} GiB including reserve\')\n    return available\n\n\ndef _design(groups, group_names, rows, path):\n    width = sum(len(groups[name][\'names\']) for name in group_names)\n    out = np.lib.format.open_memmap(path, mode=\'w+\', dtype=np.float32, shape=(len(rows),width))\n    for start in range(0,len(rows),2048):\n        ix = rows[start:start+2048]\n        col = 0\n        for name in group_names:\n            values = groups[name][\'values\']\n            out[start:start+len(ix),col:col+values.shape[1]] = values[ix]\n            col += values.shape[1]\n    out.flush()\n    return out\n\n\ndef _check_scores(score, n):\n    if np.asarray(score).shape != (n,) or not np.isfinite(score).all() or ((score < 0) | (score > 1)).any():\n        raise ValueError(\'Expected aligned finite probabilities\')\n\n\ndef _fit_arm(prepared, groups, arm, *, min_free_gib):\n    import lightgbm as lgb\n    root = Path(prepared[\'path\'])/arm\n    root.mkdir(exist_ok=True)\n    ti,vi = prepared[\'train_indices\'], prepared[\'validation_indices\']\n    group_names = ARMS[arm]\n    width = sum(len(groups[n][\'names\']) for n in group_names)\n    memory_guard((len(ti)+len(vi))*width*4*6, min_free_gib)\n    identity = dict(arm=arm, preparation_sha256=prepared[\'preparation\'][\'sha256\'],\n                    groups={n:dict(names=groups[n][\'names\'],full_sha256=groups[n][\'full_sha256\'],\n                                   provenance=groups[n][\'provenance\']) for n in group_names},\n                    settings=SETTINGS, implementation=SOURCE_SHA256,\n                    lightgbm_version=lgb.__version__, numpy_version=np.__version__)\n    receipt_path = root/\'receipt.json\'\n    if receipt_path.exists():\n        receipt = json.loads(receipt_path.read_text())\n        if receipt[\'identity\'] != identity:\n            raise ValueError(\'Saved arm identity differs; use a new version folder\')\n        for name, sha in receipt[\'files\'].items():\n            if _file_hash(root/name) != sha:\n                raise ValueError(\'Saved arm artifact hash mismatch\')\n        return {s:np.load(root/(s+\'.npy\'),allow_pickle=False) for s in (\'train\',\'validation\')}, receipt\n    xt = xv = None\n    started = time.perf_counter()\n    try:\n        xt = _design(groups,group_names,ti,root/\'train_design.npy\')\n        xv = _design(groups,group_names,vi,root/\'validation_design.npy\')\n        meta = prepared[\'metadata\']\n        y = meta.RESP.to_numpy(dtype=np.int8)\n        w = modeling.sample_weights(meta.PATIENT_ID.to_numpy()[ti], SETTINGS[\'weighting\'])\n        w = w*np.where(y[ti] == 1, modeling.class_weight(y[ti],w,SETTINGS[\'balance\']),1.)\n        params = {k:v for k,v in SETTINGS.items() if k not in {\'seeds\',\'primary_fraction\',\'weighting\',\'balance\'}}\n        model = lgb.LGBMClassifier(**params, objective=\'binary\',metric=\'None\',random_state=42,\n                                   deterministic=True,force_col_wise=True,verbosity=-1)\n        model.fit(xt,y[ti],sample_weight=w)\n        if list(model.classes_) != [0,1]:\n            raise ValueError(\'Unexpected class order\')\n        predictions = {}\n        for split,design in [(\'train\',xt),(\'validation\',xv)]:\n            scores = np.empty(len(design),np.float64)\n            for start in range(0,len(design),4096):\n                memory_guard(0,min_free_gib)\n                scores[start:start+4096] = model.booster_.predict(design[start:start+4096],num_threads=2)\n            _check_scores(scores,len(design))\n            predictions[split] = scores\n            _atomic_bytes(root/(split+\'.npy\'), workflow.array_bytes(scores))\n        _atomic_bytes(root/\'model.txt\',model.booster_.model_to_string().encode())\n        receipt = dict(identity=identity,elapsed_seconds=time.perf_counter()-started,\n                       files={n:_file_hash(root/n) for n in (\'train.npy\',\'validation.npy\',\'model.txt\')})\n        _atomic_json(receipt_path,receipt)\n        return predictions,receipt\n    finally:\n        for array in (xt,xv):\n            if array is not None and hasattr(array,\'_mmap\'):\n                array._mmap.close()\n        gc.collect()\n\n\ndef _overlap(y,a,b):\n    k = int(np.ceil(.1*len(y)))\n    positive = np.asarray(y) == 1\n    sa = set(np.lexsort((np.arange(len(y)),-np.asarray(a)))[:k])\n    sb = set(np.lexsort((np.arange(len(y)),-np.asarray(b)))[:k])\n    positives = set(np.flatnonzero(positive))\n    return dict(capacity=k,positive_gained=len((sa-sb)&positives),\n                positive_lost=len((sb-sa)&positives),positive_shared=len(sa&sb&positives),\n                selected_shared=len(sa&sb),additional_captures=len(sa&positives)-len(sb&positives))\n\n\ndef _report(prepared, predictions, groups, receipts, blocked, bootstrap):\n    meta=prepared[\'metadata\']\n    rows=[]\n    for arm,by_split in predictions.items():\n        for split,idx in [(\'train\',prepared[\'train_indices\']),(\'validation\',prepared[\'validation_indices\'])]:\n            m=meta.iloc[idx]\n            score=by_split[split]\n            _check_scores(score,len(m))\n            row=dict(Model=arm,Split=split.upper(),Snapshots=len(m),Patients=m.PATIENT_ID.nunique(),\n                     Positive_snapshots=int(m.RESP.sum()))\n            for fraction in (.1,.2,.3):\n                metric=comparison.topk_metrics(m.RESP.to_numpy(),score,fraction)\n                suffix=str(round(fraction*100))+\'%\'\n                row.update({\'Lift@\'+suffix:metric[\'lift\'],\'Captured@\'+suffix:metric[\'selected_positives\'],\n                            \'Selected@\'+suffix:metric[\'selected\']})\n            rows.append(row)\n    vi=prepared[\'validation_indices\']\n    val=meta.iloc[vi]\n    pairs={};overlap={}\n    if \'AUTOML\' in predictions:\n        baseline=predictions[\'AUTOML\'][\'validation\']\n        for arm in predictions:\n            if arm==\'AUTOML\':continue\n            scores=predictions[arm][\'validation\']\n            pairs[arm]=workflow.paired_lift_bootstrap(val.RESP.to_numpy(),scores,baseline,val.PATIENT_ID.to_numpy(),\n                                                    n_bootstrap=bootstrap,seed=20261013)\n            overlap[arm]=_overlap(val.RESP.to_numpy(),scores,baseline)\n    p=prepared[\'preparation\']\n    report=dict(schema=SCHEMA,identity=dict(version=Path(prepared[\'path\']).name,\n                evaluation_keys_sha256=p[\'evaluation_keys_sha256\'],evaluation_labels_sha256=p[\'evaluation_labels_sha256\'],\n                training_keys_sha256=p[\'training_keys_sha256\'],preparation_hash=p[\'sha256\'],settings=SETTINGS,\n                train_snapshots=len(prepared[\'train_indices\']),validation_snapshots=len(vi),\n                feature_names={n:g[\'names\'] for n,g in groups.items()},\n                feature_definition_hash=workflow.digest({n:dict(names=g[\'names\'],provenance=g[\'provenance\']) for n,g in groups.items()}),\n                protocol_hash=workflow.digest(dict(settings=SETTINGS,implementation=SOURCE_SHA256)),\n                source_identity=p[\'source_identity\'],parent_metadata_sha256=p[\'parent_metadata_sha256\']),\n                rows=rows,paired_lift=pairs,overlaps=overlap,blocked_arms=blocked,\n                population=population_summary(meta).to_dict(\'records\'),\n                completed_chunks=len(p[\'frozen_chunks\']),fit_receipts=receipts,\n                provenance={n:g[\'provenance\'] for n,g in groups.items()},\n                scope=\'Exploratory development validation on a nonrandom completed-patient prefix; TEST unused\',\n                limitations=[\'Repeated development validation is not an independent confirmation\',\n                             \'Latest source tables and refreshed checkpoints do not establish historical availability\',\n                             \'Intervals condition on fitted scores and do not include refit variation\'],\n                metric_rule=\'Snapshot lift; exact ceil(fraction*N), descending scores, frozen input-order ties\')\n    root=Path(prepared[\'path\'])\n    _atomic_json(root/\'report.json\',report)\n    _atomic_json(root/\'report_receipt.json\',dict(schema=SCHEMA,sha256=_file_hash(root/\'report.json\')))\n    return report\n\n\ndef run(prepared, timing, nlp=None, *, bootstrap=1000, min_free_gib=2):\n    """Run static/timing first. A second call may add verified NLP without refitting them."""\n    if bootstrap < 20:\n        raise ValueError(\'At least twenty bootstrap draws are required\')\n    memory_guard(0,min_free_gib)\n    p=prepared[\'preparation\']\n    if _load_preparation(Path(prepared[\'path\'])/\'prepared.json\') != p:\n        raise ValueError(\'Preparation changed since construction\')\n    meta=inputs.normalize_snapshots(prepared[\'metadata\'])\n    if (prepared[\'feature_names\'] != p[\'feature_names\']\n        or _records(meta.loc[meta.SPLIT.eq(\'train\')]) != p[\'training_records\']\n        or _records(meta.loc[meta.SPLIT.eq(\'validation\')]) != p[\'validation_records\']\n        or meta.SPLIT.eq(\'test\').any()\n        or not np.array_equal(prepared[\'train_indices\'],np.flatnonzero(meta.SPLIT.eq(\'train\')))\n        or not np.array_equal(prepared[\'validation_indices\'],np.flatnonzero(meta.SPLIT.eq(\'validation\')))\n        or workflow.array_sha256(prepared[\'X_automl\']) != p[\'automl_sha256\']):\n        raise ValueError(\'Prepared population, split indices or AutoML values changed\')\n    groups={\'automl\':dict(values=prepared[\'X_automl\'],names=prepared[\'feature_names\'],\n                          full_sha256=p[\'automl_sha256\'],evaluation_sha256=p[\'validation_automl_sha256\'],\n                          provenance=dict(verified=True,fit_scope=\'pilot_train_frequency\',cap=p[\'max_features\']))}\n    groups[\'time\']=_align_group(prepared,timing,\'time\')\n    blocked={}\n    if nlp is None:\n        blocked={arm:\'Verified matching NLP artifacts unavailable\' for arm in ARMS if \'nlp\' in ARMS[arm]}\n    else:\n        try:\n            groups[\'nlp\']=_align_group(prepared,nlp,\'nlp\')\n        except (ValueError,FileNotFoundError) as exc:\n            blocked={arm:str(exc) for arm in ARMS if \'nlp\' in ARMS[arm]}\n    definition_path=Path(prepared[\'path\'])/\'groups.json\'\n    definitions={n:dict(names=g[\'names\'],provenance=g[\'provenance\'],evaluation_sha256=g[\'evaluation_sha256\'])\n                 for n,g in groups.items()}\n    if prepared[\'previous_version\'] is not None:\n        prior_path=Path(prepared[\'previous_version\'])\n        if prior_path.is_file():prior_path=prior_path.parent\n        prior_groups=json.loads((prior_path/\'groups.json\').read_text())\n        for name,current in definitions.items():\n            if name not in prior_groups or prior_groups[name] != current:\n                raise ValueError(\'Refresh changed frozen feature definitions or validation values\')\n    if definition_path.exists():\n        old=json.loads(definition_path.read_text())\n        for name,value in old.items():\n            if name in definitions and definitions[name]!=value:\n                raise ValueError(\'Feature group changed within a version\')\n        definitions=dict(old,**definitions)\n    _atomic_json(definition_path,definitions)\n    predictions={};receipts={}\n    report=None\n    for arm,group_names in ARMS.items():\n        if arm in blocked:continue\n        score,receipt=_fit_arm(prepared,groups,arm,min_free_gib=min_free_gib)\n        predictions[arm]=score;receipts[arm]=receipt\n        report=_report(prepared,predictions,groups,receipts,blocked,bootstrap)\n        print(arm,\'saved; validation Lift@10%:\',next(r[\'Lift@10%\'] for r in report[\'rows\']\n                                                   if r[\'Model\']==arm and r[\'Split\']==\'VALIDATION\'))\n    return report\n', 'artifacts': '"""Small, immutable handoff packages for independent cached-feature analysis."""\nimport hashlib\nimport json\nimport os\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\n\n\n\ndef file_hash(path):\n    digest = hashlib.sha256()\n    with open(path, \'rb\') as stream:\n        for part in iter(lambda: stream.read(1024 * 1024), b\'\'):\n            digest.update(part)\n    return digest.hexdigest()\n\n\ndef atomic_json(path, value):\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    temp = path.with_name(path.name + \'.partial\')\n    temp.write_text(json.dumps(value, sort_keys=True, indent=2, allow_nan=False), encoding=\'utf-8\')\n    os.replace(temp, path)\n\n\ndef _verify_files(root, receipt):\n    for name, digest in receipt[\'files\'].items():\n        if Path(name).name != name or file_hash(Path(root) / name) != digest:\n            raise ValueError(\'Handoff content changed: \' + name)\n\n\ndef _verify_count_receipt(root, receipt):\n    contract = receipt.get(\'identity\', {}).get(\'contract\', {})\n    identity = receipt.get(\'identity\', {})\n    channels = identity.get(\'channel_names\', [])\n    indices = identity.get(\'channel_indices\', [])\n    if (receipt.get(\'verified\') is not True\n            or contract.get(\'orientation\') != \'0=newest\'\n            or contract.get(\'date_policy\') != \'month_end_history\'\n            or len(channels) != 2 or len(set(channels)) != 2\n            or len(indices) != 2 or len(set(indices)) != 2\n            or any(type(i) is not int or i < 0 for i in indices)\n            or set(receipt.get(\'files\', {})) != {\'counts.npy\', \'count_mask.npy\', \'count_metadata.csv\'}):\n        raise ValueError(\'Count handoff receipt or temporal contract is invalid\')\n    _verify_files(root, receipt)\n\n\ndef freeze_count_channels(counts, root, exact=None, row_block=1024):\n    """Copy only the two prespecified channels; never copy the full count tensor."""\n    if type(row_block) is not int or not 1 <= row_block <= 8192:\n        raise ValueError(\'row_block must be between 1 and 8192\')\n    root = Path(root)\n    root.mkdir(parents=True, exist_ok=True)\n    meta = inputs.normalize_snapshots(counts[\'metadata\'])\n    contract = counts[\'contract\']\n    if contract.get(\'orientation\') != \'0=newest\' or contract.get(\'date_policy\') != \'month_end_history\':\n        raise ValueError(\'Count date grid must be verified before the handoff\')\n    cols = time_lgbm.channels(counts[\'features\'], exact or {\'study\': \'\', \'diagnosis\': \'\'})\n    x, mask = counts[\'X\'], counts[\'valid\']\n    if x.shape != (len(meta), 12, len(counts[\'features\'])) or mask.shape != (len(meta), 12) or mask.dtype != bool:\n        raise ValueError(\'Count shape or mask differs from its dictionary/population\')\n    meta_hash = inputs.frame_hash(meta)\n    if contract.get(\'metadata_sha256\') != meta_hash:\n        raise ValueError(\'Count metadata differs from its saved contract\')\n    identity = dict(metadata_sha256=meta_hash, contract=contract,\n                    channel_names=[counts[\'features\'][c] for c in cols], channel_indices=cols)\n    receipt_path = root / \'counts.json\'\n    if receipt_path.exists():\n        receipt = json.loads(receipt_path.read_text(encoding=\'utf-8\'))\n        if receipt[\'identity\'] != identity:\n            raise ValueError(\'Use a different handoff directory for a different count source\')\n        _verify_count_receipt(root, receipt)\n        return str(receipt_path)\n    outputs = [(\'counts.npy\', (len(meta), 12, 2), np.float32), (\'count_mask.npy\', mask.shape, bool)]\n    for name, shape, dtype in outputs:\n        temp = root / (name + \'.partial\')\n        target = np.lib.format.open_memmap(temp, mode=\'w+\', dtype=dtype, shape=shape)\n        try:\n            for start in range(0, len(meta), row_block):\n                stop = min(start + row_block, len(meta))\n                block = x[start:stop, :, cols] if name == \'counts.npy\' else mask[start:stop]\n                if name == \'counts.npy\' and (not np.isfinite(block).all() or (block < 0).any()\n                        or (block > np.finfo(np.float32).max).any()):\n                    raise ValueError(\'Invalid monthly count values\')\n                target[start:stop] = block\n            target.flush()\n        finally:\n            target._mmap.close()\n        os.replace(temp, root / name)\n    meta.to_csv(root / \'count_metadata.csv.partial\', index=False, date_format=\'%Y-%m-%d\')\n    os.replace(root / \'count_metadata.csv.partial\', root / \'count_metadata.csv\')\n    receipt = dict(identity=identity, shape=[len(meta), 12, 2], verified=True,\n                   files={name: file_hash(root / name) for name in (\'counts.npy\', \'count_mask.npy\', \'count_metadata.csv\')},\n                   source_contract_status=\'Inherited count contract; copied subset independently hashed\',\n                   source_availability=\'Retrospective event-date bounds; ingestion-time availability unresolved\')\n    atomic_json(receipt_path, receipt)\n    return str(receipt_path)\n\n\ndef timing_group(receipt_path, metadata):\n    path = Path(receipt_path)\n    receipt = json.loads(path.read_text(encoding=\'utf-8\'))\n    _verify_count_receipt(path.parent, receipt)\n    parent = inputs.normalize_snapshots(pd.read_csv(path.parent / \'count_metadata.csv\', dtype={\'PATIENT_ID\': str}))\n    meta = inputs.normalize_snapshots(metadata)\n    if inputs.frame_hash(parent) != receipt[\'identity\'][\'metadata_sha256\']:\n        raise ValueError(\'Count metadata fingerprint changed\')\n    lookup = pd.MultiIndex.from_frame(parent[[\'PATIENT_ID\', \'END_DT\']])\n    idx = lookup.get_indexer(pd.MultiIndex.from_frame(meta[[\'PATIENT_ID\', \'END_DT\']]))\n    if (idx < 0).any() or not np.array_equal(parent.iloc[idx].RESP.to_numpy(), meta.RESP.to_numpy()) or not np.array_equal(parent.iloc[idx].SPLIT.to_numpy(), meta.SPLIT.to_numpy()):\n        raise ValueError(\'Timing keys, labels or split assignments do not match\')\n    x = np.load(path.parent / \'counts.npy\', mmap_mode=\'r\', allow_pickle=False)\n    mask = None\n    try:\n        mask = np.load(path.parent / \'count_mask.npy\', mmap_mode=\'r\', allow_pickle=False)\n        if (receipt.get(\'shape\') != [len(parent), 12, 2] or x.shape != (len(parent), 12, 2)\n                or x.dtype != np.float32 or mask.shape != (len(parent), 12) or mask.dtype != bool):\n            raise ValueError(\'Count subset shape or mask differs from its receipt\')\n        values = time_lgbm.temporal_columns(x, mask, idx, [0, 1])\n    finally:\n        x._mmap.close()\n        if mask is not None:\n            mask._mmap.close()\n    return dict(values=values, metadata=meta, names=list(time_lgbm.FEATURE_NAMES),\n                provenance=dict(verified=True, fit_scope=\'fixed_formulas\', identity=receipt[\'identity\'],\n                                receipt_sha256=file_hash(path), incomplete_history=\'all nine values missing\'))\n\n\ndef _nlp_request_identity(parent, row_block=4096):\n    """Hash the original ordered document list without a full twelve-step grid.\n\n    Patient-complete batches reuse the shared date construction and preserve\n    deduplication across overlapping snapshots. The streamed JSON bytes have\n    exactly the same meaning as inputs.stable_hash over the original list.\n    """\n    if type(row_block) is not int or row_block < 1:\n        raise ValueError(\'NLP verification row block must be positive\')\n    ordered = parent.sort_values([\'PATIENT_ID\', \'END_DT\'], kind=\'stable\').reset_index(drop=True)\n    patients = ordered.PATIENT_ID.to_numpy()\n    boundaries = np.r_[np.flatnonzero(patients[1:] != patients[:-1]) + 1, len(ordered)]\n    membership = ordered.groupby(\'PATIENT_ID\').SPLIT.first().to_dict()\n    digest = hashlib.sha256(b\'[\')\n    total = train_count = start = 0\n    while start < len(ordered):\n        position = np.searchsorted(boundaries, start + row_block, side=\'right\') - 1\n        end = int(boundaries[position]) if position >= 0 and boundaries[position] > start else int(boundaries[np.searchsorted(boundaries, start, side=\'right\')])\n        requests, grid = inputs.make_requests(ordered.iloc[start:end], date_policy=\'month_end_history\')\n        del grid\n        for patient, date in requests.itertuples(index=False, name=None):\n            split = membership[patient]\n            record = [total, patient, date.strftime(\'%Y-%m-%d\'), split]\n            if total:\n                digest.update(b\',\')\n            digest.update(json.dumps(record, sort_keys=True, separators=(\',\', \':\'), allow_nan=False, default=str).encode())\n            train_count += int(split == \'train\')\n            total += 1\n        start = end\n    digest.update(b\']\')\n    return train_count, digest.hexdigest()\n\n\ndef nlp_group(package, metadata):\n    """Validate the existing TRAIN-fitted encoder, then summarize only pilot rows."""\n    if package.get(\'_payload_verified\') is not True:\n        raise ValueError(\'Load NLP through the existing payload-verifying package reader first\')\n    parent = inputs.normalize_snapshots(package[\'metadata\'])\n    meta = inputs.normalize_snapshots(metadata)\n    manifest = package[\'manifest\']\n    names = list(package[\'feature_names\'])\n    if not names or len(set(names)) != len(names) or any(not isinstance(n, str) or not n for n in names):\n        raise ValueError(\'NLP vocabulary must be nonempty and unique\')\n    if manifest.get(\'fit_scope\') != \'train_only\' or manifest.get(\'orientation\') != \'0=newest\':\n        raise ValueError(\'NLP fitting scope or temporal orientation is not verified\')\n    if manifest.get(\'metadata_hash\') != inputs.frame_hash(parent) or names != manifest.get(\'feature_names\'):\n        raise ValueError(\'NLP parent metadata/vocabulary mismatch\')\n    train_hash = inputs.train_patient_hash(parent)\n    train_document_count, request_hash = _nlp_request_identity(parent)\n    for evidence in (manifest, manifest.get(\'fitted_state\', {}), manifest.get(\'receipt\', {})):\n        if evidence.get(\'fit_scope\', \'train_only\' if evidence is manifest else None) != \'train_only\' or evidence.get(\'train_patients_sha256\') != train_hash or evidence.get(\'train_document_count\') != train_document_count:\n            raise ValueError(\'NLP TRAIN fitting provenance is incomplete\')\n    fitted = manifest[\'fitted_state\']\n    if fitted.get(\'terms\') != names or fitted.get(\'term_order_sha256\') != inputs.stable_hash(names):\n        raise ValueError(\'NLP fitted term order differs\')\n    idf = np.asarray(fitted.get(\'idf\', []), dtype=np.float64)\n    frequency = np.asarray(fitted.get(\'document_frequency\', []), dtype=np.float64)\n    if (fitted.get(\'schema\') != \'monthly_tfidf_v1\' or fitted.get(\'training_documents\') != train_document_count\n            or idf.shape != (len(names),) or frequency.shape != idf.shape\n            or not np.isfinite(idf).all() or not np.isfinite(frequency).all()\n            or (frequency < 0).any() or (frequency > train_document_count).any()\n            or (frequency != np.floor(frequency)).any()\n            or not np.allclose(idf, np.log((1+train_document_count)/(1+frequency))+1, rtol=1e-12, atol=1e-12)):\n        raise ValueError(\'NLP fitted IDF does not match its TRAIN document frequencies\')\n    if fitted.get(\'extraction_identity\', {}).get(\'requests\') != request_hash:\n        raise ValueError(\'NLP source document identity differs\')\n    lookup = pd.MultiIndex.from_frame(parent[[\'PATIENT_ID\', \'END_DT\']])\n    idx = lookup.get_indexer(pd.MultiIndex.from_frame(meta[[\'PATIENT_ID\', \'END_DT\']]))\n    if (idx < 0).any() or not np.array_equal(parent.iloc[idx].RESP.to_numpy(), meta.RESP.to_numpy()) or not np.array_equal(parent.iloc[idx].SPLIT.to_numpy(), meta.SPLIT.to_numpy()):\n        raise ValueError(\'NLP keys, labels or split assignments do not match\')\n    x, mask = package[\'X_sequence\'], package[\'mask\']\n    if x.shape != (len(parent), 12, len(names)) or mask.shape != (len(parent), 12) or mask.dtype != bool:\n        raise ValueError(\'NLP sequence shape/mask differs\')\n    values = np.empty((len(meta), len(names) + 1), dtype=np.float32)\n    for start in range(0, len(idx), 256):\n        rows = idx[start:start+256]\n        block, observed = x[rows], mask[rows]\n        if not np.isfinite(block).all() or (block < 0).any() or (block[~observed] != 0).any():\n            raise ValueError(\'NLP values or inactive documents are invalid\')\n        means = block.mean(axis=1, dtype=np.float64)\n        if not np.isfinite(means).all() or (means > np.finfo(np.float32).max).any():\n            raise ValueError(\'NLP mean cannot be represented as finite float32 values\')\n        values[start:start+len(rows), :-1] = means\n        values[start:start+len(rows), -1] = observed.mean(axis=1)\n    return dict(values=values, metadata=meta, names=[\'NLP_MEAN_\' + n for n in names] + [\'NLP_ACTIVE_DOCUMENT_FRACTION\'],\n                provenance=dict(verified=True, fit_scope=\'train_only\', fitted_on=\'full_parent_train\',\n                    train_patients_sha256=train_hash, manifest_sha256=inputs.stable_hash(manifest),\n                    definition=\'Mean over all twelve positions, inactive documents zero; activity fraction separate\'))\n\n\ndef save_handoff(root, cache_manifest, count_receipt, metadata, config, implementation, run_prefix):\n    """Write only noncredential configuration; no source checkpoint is changed."""\n    root = Path(root)\n    def reject_credentials(value):\n        if isinstance(value, dict):\n            for key, item in value.items():\n                if any(term in str(key).lower() for term in (\'password\', \'secret\', \'token\', \'sf_options\')):\n                    raise ValueError(\'Credentials cannot be included in the handoff\')\n                reject_credentials(item)\n        elif isinstance(value, (list, tuple)):\n            for item in value:\n                reject_credentials(item)\n    reject_credentials(config)\n    meta = inputs.normalize_snapshots(metadata)\n    cache_path, count_path = Path(cache_manifest), Path(count_receipt)\n    cached = json.loads(cache_path.read_text(encoding=\'utf-8\'))\n    count = json.loads(count_path.read_text(encoding=\'utf-8\'))\n    meta_hash = inputs.frame_hash(meta)\n    if (config.get(\'raw_automl\', {}).get(\'request_steps\') != 1\n            or cached.get(\'request_steps\') != 1\n            or cached.get(\'parent_metadata_sha256\') != meta_hash\n            or cached.get(\'identity\', {}).get(\'run_identity\') != run_prefix\n            or count.get(\'identity\', {}).get(\'metadata_sha256\') != meta_hash):\n        raise ValueError(\'Handoff source identity, metadata or summary request contract differs\')\n    _verify_count_receipt(count_path.parent, count)\n    root.mkdir(parents=True, exist_ok=True)\n    meta.to_csv(root / \'parent_metadata.csv.partial\', index=False, date_format=\'%Y-%m-%d\')\n    os.replace(root / \'parent_metadata.csv.partial\', root / \'parent_metadata.csv\')\n    receipt = dict(schema=1, cache_manifest=str(cache_path.resolve()), count_receipt=str(count_path.resolve()),\n        cache_manifest_sha256=file_hash(cache_path), count_receipt_sha256=file_hash(count_path),\n        parent_metadata=str((root / \'parent_metadata.csv\').resolve()), metadata_sha256=meta_hash,\n        metadata_file_sha256=file_hash(root / \'parent_metadata.csv\'), config=config, implementation=implementation,\n        run_prefix=run_prefix, nlp_table=run_prefix+\'_NLP_INPUTS\',\n        extraction_resume=\'Resume the original extraction with its existing configuration and checkpoints\')\n    atomic_json(root / \'handoff.json\', receipt)\n    return str(root / \'handoff.json\')\n'}
PILOT_HASHES = {'inputs': 'ef7c11a1774e86cff81e3cccbb34e0a2357901bbc503e230f1a0515165aeaa09', 'workflow': '85b27568c2b0608f6f265b328deedc3402c6f9f37d0164e399aa5990bcab7d21', 'comparison': 'e253bb1ac7757ed53243bbcd0ee412da9a9836a9bab778c1d45c179b71e6255d', 'warehouse': 'ec0b61c5134b7d8f83dd07e2592c7baf7550c1e34339a59c996a228f8a944bfe', 'pipeline': 'eb0ec830d5ca1822a3d87be5a72036961e085eca387b70c81ecbab77829ce2fc', 'modeling': '3c15a23a8c188c74b02c1eda57a002c16eb3206a638ec0a93dc4774e1982d3fe', 'raw_automl': '8f9d5f06f2a2efed2d733ea3efc4277bdf3931195853fc786b5becb04b23c4ce', 'settings': '3f820e6de4cb4d37355a7a2e542d1b3dc8a0b03ae61d1c8c81794575b2ba59ad', 'time_lgbm': 'ecbffb129802863dfc4c400344a39903aa4d8007347ca3e2a0f2464102dab851', 'cache': '8c00159e774578d6cf95f55f94376f3ffe09768aec95cf07fae9bfe9cbf35514', 'analysis': 'b6db9c48b685123115262bdd37e5a0b2c86e5e08fb3b11adf44d4f5dabb1f740', 'artifacts': '3a3fbdafbb48d60d5653edef1cf0d21bd3bd84a4c4f346e2442b550dfb851dad'}

def pilot_module(name, **injected):
    source = PILOT_SOURCES[name]
    if hashlib.sha256(source.encode()).hexdigest() != PILOT_HASHES[name]:
        raise ValueError("Embedded implementation checksum mismatch")
    module = types.ModuleType("pilot_" + name)
    module.__dict__.update(injected)
    exec(source, module.__dict__)
    return module
pilot_inputs = pilot_module("inputs")
pilot_time = pilot_module("time_lgbm")
pilot_cache = pilot_module("cache")
pilot_cache.inputs = pilot_inputs
pilot_artifacts = pilot_module("artifacts", inputs=pilot_inputs, time_lgbm=pilot_time)

pilot_workflow = pilot_module("workflow")
pilot_comparison = pilot_module("comparison")
pilot_modeling = pilot_module("modeling")
pilot_raw = pilot_module("raw_automl")
pilot_settings = pilot_module("settings")
pilot_analysis = pilot_module("analysis", inputs=pilot_inputs, modeling=pilot_modeling,
    raw_automl=pilot_raw, workflow=pilot_workflow, FULL_SETTINGS=pilot_settings.SETTINGS,
    comparison=pilot_comparison, cache=pilot_cache)
pilot_analysis.SOURCE_SHA256 = hashlib.sha256(json.dumps(PILOT_HASHES, sort_keys=True).encode()).hexdigest()
pilot_warehouse = pilot_module("warehouse")
pilot_pipeline = pilot_module("pipeline")


In [ ]:
handoff_path = Path(HANDOFF_MANIFEST)
if not handoff_path.is_file():
    raise FileNotFoundError("Run HANDOFF_CELL.py in the paused notebook 01 session first, then resume extraction. Set HANDOFF_MANIFEST to its output.")
handoff = json.loads(handoff_path.read_text(encoding="utf-8"))
for path_field, hash_field in (("cache_manifest", "cache_manifest_sha256"), ("count_receipt", "count_receipt_sha256")):
    if pilot_artifacts.file_hash(handoff[path_field]) != handoff[hash_field]:
        raise ValueError("Handoff artifact checksum mismatch: " + path_field)
if pilot_artifacts.file_hash(handoff["parent_metadata"]) != handoff["metadata_file_sha256"]:
    raise ValueError("Handoff metadata checksum mismatch")
parent_metadata = pilot_inputs.normalize_snapshots(pd.read_csv(handoff["parent_metadata"], dtype={"PATIENT_ID":str}))
if pilot_inputs.frame_hash(parent_metadata) != handoff["metadata_sha256"]:
    raise ValueError("Handoff population identity changed")
pilot_analysis.memory_guard(0, MIN_FREE_GIB)
prepared = pilot_analysis.prepare(handoff["cache_manifest"], PILOT_OUTPUT,
    previous_version=PREVIOUS_PREPARED or None, max_features=512)
display(pilot_analysis.population_summary(prepared["metadata"]))
print("Complete patients from a sorted request prefix; exploratory development subset, not a random sample.")
print("TEST is unused. Validation keys and feature definitions remain fixed on refresh.")
print("AutoML columns:", len(prepared["feature_names"]))


In [ ]:
timing = pilot_artifacts.timing_group(handoff["count_receipt"], prepared["metadata"])
print("Timing columns:", timing["names"])
print("Rows with complete monthly history:", int(np.isfinite(timing["values"]).all(axis=1).sum()))
report = pilot_analysis.run(prepared, timing, bootstrap=BOOTSTRAP_DRAWS, min_free_gib=MIN_FREE_GIB)
display(pd.DataFrame(report["rows"]))
print("Saved report:", str(Path(PILOT_OUTPUT)/"report.json"))


In [ ]:
nlp = None
nlp_blocker = None
if ENABLE_NLP:
    if not all(sf_options.values()):
        nlp_blocker = "Connection settings are empty; matching NLP package not loaded"
    else:
        pilot_warehouse.spark = spark
        pilot_warehouse.sf_options_dl_poc = dict(sf_options)
        package = None
        try:
            pilot_analysis.memory_guard(0, MIN_FREE_GIB)
            package = pilot_pipeline.load_nlp(parent_metadata, handoff["config"], handoff["implementation"],
                handoff["run_prefix"], pilot_inputs, pilot_workflow, pilot_warehouse)
            package["_payload_verified"] = True
            nlp = pilot_artifacts.nlp_group(package, prepared["metadata"])
        except Exception as error:
            nlp_blocker = type(error).__name__ + ": matching NLP package unavailable or provenance failed; inspect the exception locally"
            print(nlp_blocker)
        finally:
            if package is not None:
                pilot_pipeline.release_nlp(package)
    if nlp is not None:
        print("NLP encoder was fitted on full parent TRAIN patients; monthly means summarize twelve positions.")
        report = pilot_analysis.run(prepared, timing, nlp, bootstrap=BOOTSTRAP_DRAWS, min_free_gib=MIN_FREE_GIB)
if nlp_blocker:
    pilot_artifacts.atomic_json(Path(PILOT_OUTPUT)/"nlp_blocker.json", {"reason":nlp_blocker})
    print(nlp_blocker, "; completed AutoML/timing fits remain available.")


In [ ]:
import matplotlib.pyplot as plt
rows = pd.DataFrame(report["rows"])
validation = rows.loc[rows.Split.eq("VALIDATION")].sort_values("Lift@10%", ascending=False, kind="stable").copy()
validation.insert(0, "Rank", validation["Lift@10%"].rank(ascending=False, method="min").astype(int))
display(validation[["Rank","Model","Lift@10%","Captured@10%","Selected@10%","Lift@20%","Lift@30%"]])
paired = []
for arm, result in report["paired_lift"].items():
    for fraction, delta in result["lift_by_fraction"].items():
        paired.append(dict(Model=arm, Compared_with="AUTOML", **delta))
display(pd.DataFrame(paired))
display(pd.DataFrame(report["overlaps"]).T)
fig, axes = plt.subplots(1, 2, figsize=(13, 4), constrained_layout=True)
validation.plot.barh(x="Model", y="Lift@10%", ax=axes[0], legend=False, color="#0B6E75")
axes[0].invert_yaxis()
axes[0].axvline(1, color="#777777", linestyle="--")
axes[0].set_title("Development validation: lift at 10%")
rows.pivot(index="Model", columns="Split", values="Lift@10%").plot.bar(ax=axes[1], color=["#A5B4C2", "#0B6E75"])
axes[1].set_title("TRAIN versus validation: overfitting diagnostic")
axes[1].tick_params(axis="x", labelrotation=25)
plt.show()
fig.savefig(Path(PILOT_OUTPUT)/"lift_diagnostics.png", dpi=160, bbox_inches="tight")
print(report["scope"])
print("Additional capture is measured against AutoML on identical rows; historical production scores are excluded.")
print("Blocked arms:", report["blocked_arms"])
print("Refresh: create a new handoff version with PREVIOUS_CACHE_DIRECTORY, then set a new PILOT_OUTPUT and PREVIOUS_PREPARED to this version's prepared.json.")
print("Reuse completed fits after interruption by keeping the same handoff and PILOT_OUTPUT. No extraction or other notebook is rerun here.")
